In [1]:
from dotenv import load_dotenv
load_dotenv()

%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
%gui qt

from copy import deepcopy
import time
import xarray as xr # Assuming you're using this
import numpy as np   # For the example
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import xarray as xr
# import zarr
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
# import panel as pn
# pn.extension()

import IPython
# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pyqtgraph as pg
import phopymnehelper.type_aliases as types
from pypho_timeline.widgets.simple_timeline_widget import SimpleTimelineWidget
from pypho_timeline.widgets.TimelineWindow.MainTimelineWindow import MainTimelineWindow
from pypho_timeline.__main__ import VideoTrackDatasource

# CustomGraphicsLayoutWidget
from pypho_timeline.xdf_session_discovery import discover_xdf_files_for_timeline
from pypho_timeline.timeline_builder import TimelineBuilder

def get_now_time_str(time_separator='-') -> str:
    return str(time.strftime(f"%Y-%m-%d_%H{time_separator}%m", time.localtime(time.time())))

# Create Qt application
app = pg.mkQApp("pyPhoTimelineTestingNotebookExample")
builder: TimelineBuilder = TimelineBuilder()

Automatic pdb calling has been turned OFF


c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\core\time_synchronized_plotter_base.py:3: PythonQtWarning: Selected binding 'pyqt5' could not be found; falling back to 'pyqt6'
  from qtpy import QtCore, QtWidgets


Using matplotlib as 2D backend.


C:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\PhoPyMNEHelper\src\phopymnehelper\helpers\indexing_helpers.py:1078: UserWarning: registration of accessor <class 'phopymnehelper.helpers.indexing_helpers.PhoDataframeAccessor'> under name 'pho' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  @pd.api.extensions.register_dataframe_accessor("pho")
C:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\NeuroPy\neuropy\utils\mixins\time_slicing.py:404: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  @pd.api.extensions.register_dataframe_accessor("time_point_event")


2026-04-29 14:45:03 - pypho_timeline - INFO - [MainThread] - Logging configured: level=DEBUG, console=True, file=False, log_file=N/A


In [2]:
# n_most_recent_sessions_to_preprocess: int = None # None means all sessions
# n_most_recent_sessions_to_preprocess: int = 100
n_most_recent_sessions_to_preprocess: int = 15
# n_most_recent_sessions_to_preprocess: int = 8
# n_most_recent_sessions_to_preprocess: int = 5
# n_most_recent_sessions_to_preprocess = None

# enable_video_track: bool = True
enable_video_track: bool = False

# Optional: only include streams whose name matches any of these patterns (regex).
STREAM_ALLOWLIST: Optional[List[str]] = None  # e.g. [r"EEG.*", r"MOTION.*"]

# Optional: exclude streams whose name matches any of these patterns (regex).
# STREAM_BLOCKLIST: Optional[List[str]] = ['Epoc X Motion', 'Epoc X eQuality']
STREAM_BLOCKLIST: Optional[List[str]] = ['Epoc X eQuality', 'VideoRecorderMarkers']


# db_root_path = Path('/content/drive/MyDrive/Databases')
# db_root_path = Path(r'E:/Dropbox (Personal)/Databases') ## APOGEE
db_root_path = Path(r'E:/Dropbox (Personal)/Databases') # WIN10_VM
assert db_root_path.exists(), f"'{db_root_path.as_posix()}' does not exist!"

pho_log_to_LSL_recordings_path: Path = db_root_path.joinpath('UnparsedData/PhoLogToLabStreamingLayer_logs')
## These contain little LSL .fif files with names like: '20250808_062814_log.fif',

video_recordings_path: Path = db_root_path.joinpath('UnparsedData/LabRecorderStudies/sub-P001/Videos')

# eeg_analyzed_parent_export_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed')
# pickled_data_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed/PICKLED_COLLECTION')
# assert pickled_data_path.exists()
xdf_to_rerun_rrd_parent_export_path = db_root_path.joinpath('AnalysisData/to_rerun_rrd').resolve()
xdf_to_rerun_rrd_parent_export_path.mkdir(exist_ok=True)
# print(f'xdf_to_rerun_rrd_parent_export_path: "{xdf_to_rerun_rrd_parent_export_path.as_posix()}"')

# eeg_analyzed_parent_export_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed')
# pickled_data_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed/PICKLED_COLLECTION')
# assert pickled_data_path.exists()
xdf_to_exported_EDF_parent_export_path = db_root_path.joinpath('AnalysisData/exported_EDF').resolve()
xdf_to_exported_EDF_parent_export_path.mkdir(exist_ok=True)
# print(f'xdf_to_rerun_rrd_parent_export_path: "{xdf_to_rerun_rrd_parent_export_path.as_posix()}"')


# lab_recorder_output_path = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001")
lab_recorder_output_path = db_root_path.joinpath('UnparsedData/LabRecorderStudies/sub-P001')
assert lab_recorder_output_path.exists()

xdf_file_cache_filename: str = f"{get_now_time_str()}_found_xdf_files.csv"
xdf_file_cache_filepath: Path = xdf_to_rerun_rrd_parent_export_path.joinpath(xdf_file_cache_filename).resolve()
print(f'exporting xdf .csv to xdf_file_cache_filepath: "{xdf_file_cache_filepath}..."')
discovery = discover_xdf_files_for_timeline(xdf_discovery_dirs=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], n_most_recent=n_most_recent_sessions_to_preprocess, csv_export_path=xdf_file_cache_filepath)
final_xdf_paths: List[Path] = discovery.xdf_paths
print(f'processing len(active_EEG_recording_files): {len(final_xdf_paths)} recording files...')

## OUTPUTS: final_xdf_paths

# ==================================================================================================================================================================================================================================================================================== #
# BEGIN MAIN                                                                                                                                                                                                                                                                           #
# ==================================================================================================================================================================================================================================================================================== #

# builder: TimelineBuilder = TimelineBuilder()
# active_video_discovery_dirs: List[Path] = [video_recordings_path] if video_recordings_path.exists() and video_recordings_path.is_dir() else []
active_video_discovery_dirs = []
builder.set_refresh_config(xdf_discovery_dirs=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], n_most_recent=n_most_recent_sessions_to_preprocess, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST, video_discovery_dirs=active_video_discovery_dirs)

exporting xdf .csv to xdf_file_cache_filepath: "E:\Dropbox (Personal)\Databases\AnalysisData\to_rerun_rrd\2026-04-29_14-04_found_xdf_files.csv..."
processing len(active_EEG_recording_files): 15 recording files...


In [ ]:
builder.refresh_from_directories()

In [3]:
timeline: SimpleTimelineWidget = builder.build_from_xdf_files(xdf_file_paths=final_xdf_paths, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST)
if timeline is None:
    print("WARN: No streams found. Check XDF paths and stream filters.")


Stream 3: Calculated effective sampling rate 31.9989 Hz is different from specified rate 16.0000 Hz.
skipping empty stream: "VideoRecorderMarkers"
skipping empty stream: "EventBoard"
skipping empty stream: "WhisperLiveLogger"
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\stream_to_datasources.py:174: RuntimeWarning: Could not get size for self.info
  logger.info(f'\traws_dict: {a_raws_dict}')
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\stream_to_datasources.py:174: RuntimeWarning: Could not get size for self.info
  logger.info(f'\traws_dict: {a_raws_dict}')
Stream 2: Calculated effective sampling rate 32.0131 Hz is different from specified rate 16.0000 Hz.
Stream 1: Calculated effective sampling rate 31.3821 Hz is different from specified rate 16.0000 Hz.
skipping empty stream: "EventBoard"
skipping empty stream: "WhisperLiveLogger"
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_

Using earliest reference datetime: 2026-04-28 03:35:13+00:00 for timestamp normalization

Processing stream 'EventBoard' from 5 file(s)...
  Skipping empty stream from LabRecorder_Apogee_2026-04-29T180046.667Z_eeg.xdf
  Skipping empty stream from LabRecorder_Apogee_2026-04-29T175706.951Z_eeg.xdf
  Skipping empty stream from LabRecorder_Apogee_2026-04-29T145857.301Z_eeg.xdf
  Skipping empty stream from LabRecorder_Apogee_2026-04-28T062408.607Z_eeg.xdf
  Skipping empty stream from LabRecorder_Apogee_2026-04-28T062133.145Z_eeg.xdf
  No valid intervals for stream 'EventBoard'

Processing stream 'Epoc X Motion' from 5 file(s)...

Processing stream 'TextLogger' from 5 file(s)...

Processing stream 'WhisperLiveLogger' from 5 file(s)...
  Skipping empty stream from LabRecorder_Apogee_2026-04-29T180046.667Z_eeg.xdf
  Skipping empty stream from LabRecorder_Apogee_2026-04-29T175706.951Z_eeg.xdf
  Skipping empty stream from LabRecorder_Apogee_2026-04-29T145857.301Z_eeg.xdf
  Skipping empty stream 

C:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\PhoPyMNEHelper\src\phopymnehelper\analysis\computations\specific\EEG_Spectograms.py:46: RuntimeWarning: No EEG channels remain for spectrogram after exclusions or none are present.
  return EEGComputations.raw_spectogram_working(raw, picks=picks, nperseg=nperseg, noverlap=noverlap, mask_bad_annotated_times=mask_bad_annotated_times)


	Bad channels: ['F7', 'FC5', 'T7', 'P7']
	Good channels: ['AF3', 'AF4', 'F3', 'F4', 'F8', 'FC6', 'O1', 'O2', 'P8', 'T8']
trying EEGComputations.detect_bad_channels_sliding_window(...) to detect bad channels...
trying EEGComputations.detect_bad_channels_sliding_window(...) to detect bad channels...
	Bad channels: ['P7']
	Good channels: ['AF3', 'AF4', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'O2', 'P8', 'T7', 'T8']
trying EEGComputations.detect_bad_channels_sliding_window(...) to detect bad channels...
	Bad channels: ['P7', 'O2', 'T8']
	Good channels: ['AF3', 'AF4', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'P8', 'T7']


C:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\PhoPyMNEHelper\src\phopymnehelper\analysis\computations\eeg_registry.py:32: RuntimeWarning: EEGComputations.detect_bad_channels_sliding_window(...) failed (ValueError: No channels supplied to apply the reference to); no channels marked bad.
  return EEGComputations.time_independent_bad_channels(ctx.raw, **dict(params))
C:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\PhoPyMNEHelper\src\phopymnehelper\analysis\computations\specific\EEG_Spectograms.py:46: RuntimeWarning: No EEG channels remain for spectrogram after exclusions or none are present.
  return EEGComputations.raw_spectogram_working(raw, picks=picks, nperseg=nperseg, noverlap=noverlap, mask_bad_annotated_times=mask_bad_annotated_times)


added_timeline_idx: 0
Connected "on_options_changed" signal from options panel to track_renderer.on_options_changed
Connected "on_options_accepted" signal from options panel to track_renderer.on_options_accepted
Connected "on_options_rejected" signal from options panel to track_renderer.on_options_rejected
Connected "on_options_changed" signal from options panel to track_renderer.on_options_changed
Connected "on_options_accepted" signal from options panel to track_renderer.on_options_accepted
Connected "on_options_rejected" signal from options panel to track_renderer.on_options_rejected
Connected "on_options_changed" signal from options panel to track_renderer.on_options_changed
Connected "on_options_accepted" signal from options panel to track_renderer.on_options_accepted
Connected "on_options_rejected" signal from options panel to track_renderer.on_options_rejected
Connected "on_options_changed" signal from options panel to track_renderer.on_options_changed
Connected "on_options_acce

t_duration_sec: 52.501779079437256
curr_df_points_per_sec: 32.01796261906822
t_duration_sec: 3227.9393973350525
curr_df_points_per_sec: 31.64774409468145
curr_downsampled_points_per_sec: 9.99966113920929
result_df
t_duration_sec: 4646.605194568634
curr_df_points_per_sec: 32.013694680554764
t_duration_sec: 208.477374792099
curr_df_points_per_sec: 32.01786288155511
t_duration_sec: 353.38775873184204
curr_df_points_per_sec: 31.384788312421655
curr_downsampled_points_per_sec: 9.996288576054061
result_df
result_df
curr_downsampled_points_per_sec: 9.997516644827853
result_df
result_df
curr_downsampled_points_per_sec: 9.999877948963091
result_df
result_df
result_df
result_df
curr_downsampled_points_per_sec: 9.999988820723052
result_df
t_duration_sec: 52.50554394721985
curr_df_points_per_sec: 128.32930569718962
curr_downsampled_points_per_sec: 9.998944121553064
result_df
t_duration_sec: 208.5108938217163
curr_df_points_per_sec: 128.43453648468738


In [ ]:
## Create another independent timeline
timeline_alt: SimpleTimelineWidget = builder.build_from_xdf_files(xdf_file_paths=final_xdf_paths, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST)
if timeline_alt is None:
    print("WARN: No streams found. Check XDF paths and stream filters.")


In [ ]:
builder.default_post_timeline_create_display_updates(timeline=timeline_alt)

In [ ]:
STREAM_BLOCKLIST_alt2: Optional[List[str]] = ['Epoc X eQuality', 'VideoRecorderMarkers', 'TextLogger', 'EventBoard', 'log_widget']
## Create another independent timeline
timeline_alt2: SimpleTimelineWidget = builder.build_from_xdf_files(xdf_file_paths=final_xdf_paths, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST_alt2)
if timeline_alt2 is None:
    print("WARN: No streams found. Check XDF paths and stream filters.")



In [ ]:
builder.default_post_timeline_create_display_updates(timeline=timeline_alt2)

### Get all timeline tracks

In [4]:
from phopymnehelper.motion_data import MotionData, MotionDataFrame, BadMotionDataFrame
from phopymnehelper.MNE_helpers import MNEHelpers

all_track_names = timeline.get_all_track_names()
print(f"all_track_names: {all_track_names}")
eeg_widget, eeg_track, eeg_ds = timeline.get_track_tuple('EEG_Epoc X')
detailed_eeg_df: pd.DataFrame = eeg_ds.detailed_df
motion_widget, motion_track, motion_ds = timeline.get_track_tuple('MOTION_Epoc X Motion')
txt_log_widget, txt_log_renderer, txt_log_ds = timeline.get_track_tuple('LOG_TextLogger')


eeg_spectogram_track_names = ['EEG_Spectrogram_Epoc X_Frontal-L',
 'EEG_Spectrogram_Epoc X_Frontal-R',
 'EEG_Spectrogram_Epoc X_Posterior-L',
 'EEG_Spectrogram_Epoc X_Posterior-R', 'EEG_Spectrogram_Epoc X_All',
]

for a_spectogram_track_name in eeg_spectogram_track_names:
    if a_spectogram_track_name in all_track_names:
        a_spectogram_widget, a_spectogram_track, a_spectogram_ds = timeline.get_track_tuple(a_spectogram_track_name)
        # a_spectogram_track
        a_root_graphics_layout_widget = a_spectogram_widget.getRootGraphicsLayoutWidget()
        a_plot_item = a_spectogram_widget.getRootPlotItem()
        a_plot_item.hideAxis('bottom')
        # a_spectogram_ds.detailed_df


## OUTPUTS: eeg_ds, motion_ds, txt_log_ds

t_duration_sec: 4646.81086063385
curr_df_points_per_sec: 128.37966035024428
t_duration_sec: 3218.8959410190582
curr_df_points_per_sec: 127.2650646389984
t_duration_sec: 367.0552797317505
curr_df_points_per_sec: 121.09892556915331
curr_downsampled_points_per_sec: 9.999477541843659
result_df
curr_downsampled_points_per_sec: 9.998493967126944
result_df
curr_downsampled_points_per_sec: 9.999701944328688
result_df
curr_downsampled_points_per_sec: 9.999976627768644
result_df
all_track_names: ['MOTION_Epoc X Motion', 'LOG_TextLogger', 'EEG_Spectrogram_Epoc X_Frontal', 'EEG_Spectrogram_Epoc X_Posterior', 'EEG_Epoc X']


mask_col 'is_valid' not found; available columns: ['AccX', 'AccX_diff', 'AccX_smooth', 'AccY', 'AccY_diff', 'AccY_smooth', 'AccZ', 'AccZ_diff', 'AccZ_smooth', 'GyroX', 'GyroY', 'GyroZ', 'is_moving', 't', 'total']
mask_col 'is_valid' not found; available columns: ['AccX', 'AccX_diff', 'AccX_smooth', 'AccY', 'AccY_diff', 'AccY_smooth', 'AccZ', 'AccZ_diff', 'AccZ_smooth', 'GyroX', 'GyroY', 'GyroZ', 'is_moving', 't', 'total']
mask_col 'is_valid' not found; available columns: ['AccX', 'AccX_diff', 'AccX_smooth', 'AccY', 'AccY_diff', 'AccY_smooth', 'AccZ', 'AccZ_diff', 'AccZ_smooth', 'GyroX', 'GyroY', 'GyroZ', 'is_moving', 't', 'total']
mask_col 'is_valid' not found; available columns: ['AccX', 'AccX_diff', 'AccX_smooth', 'AccY', 'AccY_diff', 'AccY_smooth', 'AccZ', 'AccZ_diff', 'AccZ_smooth', 'GyroX', 'GyroY', 'GyroZ', 'is_moving', 't', 'total']
mask_col 'is_valid' not found; available columns: ['AccX', 'AccX_diff', 'AccX_smooth', 'AccY', 'AccY_diff', 'AccY_smooth', 'AccZ', 'AccZ_diff', 'Acc

In [ ]:
eeg_track

In [ ]:
# dock_height = max(120, row_height_px * 4 + 28)

identifier: str = 'timeline_overview_strip'
timeline_overview_strip = timeline.ui.timeline_overview_strip
timeline_overview_strip.size()

# timeline_overview_strip.setMaximumHeight(120)

In [ ]:
dock_area: NestedDockAreaWidget = timeline.ui.dynamic_docked_widget_container
dock_area

In [ ]:
dock_area.size()

height_px: int = dock_area.height()
# dock_area.heightMM() # 283

height_px
flat_dockitems = timeline.get_flat_dockitems_list()
flat_dockitems

In [ ]:
stretches: List[Tuple] = [d.stretch() for d in flat_dockitems]
height_px: np.array = np.array([d.height() for d in flat_dockitems])
stretches_height: np.array = np.array([s[1] for s in stretches])
# height_px
# stretches_height
total_height_px: float = np.nansum(height_px)
total_height_stretches: float = np.nansum(stretches_height)

# total_height_px
# total_height_stretches
n_tracks: int = len(stretches_height)

normalized_height_stretches = (stretches_height / total_height_stretches)
track_unit_normalized_height_stretches = (normalized_height_stretches * n_tracks)

track_unit_normalized_height_stretches

In [ ]:
normalized_height_stretches

In [ ]:
normalized_height_stretches = timeline.compute_normalized_stretch_heights()
normalized_height_stretches

In [ ]:
# desired_interval_idx, (desired_t_start, desired_t_end) = timeline.find_next_active_interval(is_jump_previous=False)
desired_interval_idx, (desired_t_start, desired_t_end) = timeline.find_next_active_interval(is_jump_next=True)
desired_interval_idx, (desired_t_start, desired_t_end)
# timeline.update_window(desired_t_start, desired_t_end)


### Collapse Specific Docs

In [ ]:
all_track_names # ['MOTION_Epoc X Motion',
#  'EEG_Spectrogram_Epoc X_All',
#  'EEG_Epoc X',
#  'LOG_TextLogger']


## get all dock items
# timeline._set_track_dock_group(

In [5]:
from pypho_timeline.docking.nested_dock_area_widget import NestedDockAreaWidget
from pypho_timeline.EXTERNAL.pyqtgraph.dockarea.Dock import DockDisplayConfig, DockButtonConfig, Dock, DockLabel
from pypho_timeline.docking.dynamic_dock_display_area import DynamicDockDisplayAreaOwningMixin
from pypho_timeline.docking.specific_dock_widget_mixin import SpecificDockWidgetManipulatingMixin
from pypho_timeline.widgets.simple_timeline_widget import SimpleTimelineWidget, SimpleTimeWindow

timeline.collapse_docks_programmatically(dock_identifiers_to_collapse = ['MOTION_Epoc X Motion', 'log_widget'])

# dock_identifiers_to_collapse = ['MOTION_Epoc X Motion', 'log_widget']
# for a_dock_identifier in dock_identifiers_to_collapse:
#     a_dock: Dock = timeline.dock_container.find_display_dock(a_dock_identifier)
#     a_dock.perform_programmatic_collapse(is_collapse_active=True)
#     # a_dock.setContentVisibility(False)



In [ ]:
a_dock_identifier: str = 'LOG_EventBoard'
# a_dock_identifier: str = 'LOG_TextLogger'
a_dock: Dock = timeline.dock_container.find_display_dock(a_dock_identifier)
# a_dock.setContentVisibility(False)
# a_dock.size() # QSize(1796, 184)

## capture the original (relative) stretches:
wx, wy = a_dock.stretch() # (500, 80)
(wx, wy)

new_wy = (wy * 0.25) ## 1/4 height
a_dock.setStretch(wx, new_wy)
# a_dock.resize(



In [ ]:
# a_dock.widgetArea.setFixedHeight(40) ## this didn't work, maybe it's not the dock's actual contents though

In [ ]:
.setFixedHeight(

In [ ]:
active_intervals = timeline.find_intervals_in_active_window()
active_intervals

### Zoom to a particular event:

In [6]:
timeline.go_to_specific_interval(desired_interval_idx=-1)

In [ ]:
eeg_overview_intervals_df: pd.DataFrame = eeg_ds.get_overview_intervals()
eeg_overview_intervals_df
desired_t_start, desired_t_end = eeg_overview_intervals_df.iloc[-1][['t_start', 't_end']].to_numpy()
desired_t_start, desired_t_end
timeline.update_window(desired_t_start, desired_t_end)

In [ ]:
timeline.go_to_specific_interval(desired_interval_idx=-1)

# EEG Power Bands Track

In [ ]:
from pypho_timeline.core.synchronized_plot_mode import SynchronizedPlotMode
from pypho_timeline.rendering.datasources.specific.eeg import EEGTrackDatasource, EEGFPTrackDatasource

assert isinstance(eeg_ds, EEGTrackDatasource)

gfp_ds = EEGFPTrackDatasource(intervals_df=eeg_ds.intervals_df.copy(), eeg_df=eeg_ds.detailed_df, custom_datasource_name=f"GFP_{eeg_ds.custom_datasource_name}", channel_names=eeg_ds.channel_names, max_points_per_second=eeg_ds.max_points_per_second, enable_downsampling=eeg_ds.enable_downsampling, lab_obj_dict=getattr(eeg_ds, "lab_obj_dict", None), raw_datasets_dict=getattr(eeg_ds, "raw_datasets_dict", None))
track_widget, _root, gfp_plot, _dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(name=gfp_ds.custom_datasource_name, dockSize=(500, 120), dockAddLocationOpts=["bottom"], sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA)
ref_name = eeg_ds.custom_datasource_name
if ref_name in timeline.ui.matplotlib_view_widgets:
    ref_plot = timeline.ui.matplotlib_view_widgets[ref_name].getRootPlotItem()
    x0, x1 = ref_plot.getViewBox().viewRange()[0]
    gfp_plot.setXRange(x0, x1, padding=0)
gfp_plot.setYRange(0, 5, padding=0)
gfp_plot.hideAxis("left")
gfp_track_renderer = timeline.add_track(gfp_ds, name=gfp_ds.custom_datasource_name, plot_item=gfp_plot)

In [ ]:
gfp_track_renderer

### Other

In [ ]:
all_ds_dict = {k:timeline.track_datasources[k] for k in timeline.track_datasources.dynamically_added_attributes}
all_ds_dict

In [ ]:
# all_ds_lab_obj_dict = {k:getattr(v, 'lab_obj_dict', None) for k, v in all_ds_dict.items()}
all_ds_earliest_unix_timestamp_dict = {k:getattr(v, 'earliest_unix_timestamp', None) for k, v in all_ds_dict.items()}
all_ds_earliest_unix_timestamp_dict
# earliest_unix_timestamp
# all_ds_lab_obj_dict

In [ ]:
from datetime import datetime
from pypho_timeline.utils.datetime_helpers import unix_timestamp_to_datetime, datetime_to_unix_timestamp

now_dt = datetime.now()
now_unix_timestamp: float = datetime_to_unix_timestamp(now_dt)

now_dt
now_unix_timestamp

In [ ]:
import pandas as pd

ds = timeline.track_datasources["LOG_TextLogger"]

# Preserve the current times as UTC, then store as unix-second floats.
t_utc = pd.to_datetime(ds.detailed_df["t"], utc=True)
ds.detailed_df["t"] = (t_utc.astype("int64") / 1e9).astype(float)

# Keep the interval slicing path in the same numeric domain.
ds.intervals_df["t_start_dt"] = ds.intervals_df["t_start"].astype(float)
ds.intervals_df["t_end_dt"] = ds.intervals_df["t_end"].astype(float)

# Force cache clear + re-render.
ds.source_data_changed_signal.emit()

In [ ]:
temporal_columns = ['t_start', 't_duration', 't_end', 't_start_dt', 't_end_dt']
all_ds_intervals = {k:v.intervals_df[temporal_columns] for k, v in all_ds_dict.items()}
all_ds_intervals_df = pd.concat(list(all_ds_intervals.values()))
# all_ds_intervals


all_ds_intervals[



In [ ]:
eeg_ds.intervals_df

In [ ]:
txt_log_ds.intervals_df

In [ ]:
## hide MOTION channels we don't need
motion_track


In [ ]:
motion_widget


## Debug Excessive Layout Margins

In [ ]:
from pypho_timeline.widgets.custom_graphics_layout_widget import CustomViewBox, CustomGraphicsLayoutWidget

root_graphics_layout_widget = a_spectogram_widget.ui.root_graphics_layout_widget
root_graphics_layout_widget.setBackground(pg.mkColor('red'))

In [ ]:
root_plot_viewBox: CustomViewBox = a_spectogram_widget.ui.root_plot_viewBox
root_plot_viewBox
root_plot_viewBox.setBackgroundColor(pg.mkColor('green')) ## viewbox is too small

In [ ]:
# root_graphics_layout_widget.setYRange(0, 1, padding=0)
root_plot_viewBox.setYRange(0, 1, padding=0, update=True)
# root_plot_viewBox.setYRange(-0.05, 1.05, padding=0, update=True)



In [ ]:
a_layout = root_plot.parentItem()
# a_layout.getWindowFrameMargins()

root_graphics_layout_widget.geometry()
a_layout.geometry()
root_plot.geometry()

In [ ]:
root_plot_viewBox

In [ ]:
root_plot.setContentsMargins(0,0,0,0)

desired_geometry = (a_layout.geometry().x(), a_layout.geometry().y(), a_layout.geometry().width(), a_layout.geometry().height())
root_plot.setGeometry(*desired_geometry)


In [ ]:
root_plot.setYRange(0, 1, padding=0)
# root_plot.setYRange(0, 1, padding=0.1) ## did move it inset of the green box


In [ ]:

root_plot.getContentsMargins()
root_plot.

In [ ]:

root_plot = a_spectogram_widget.ui.root_plot
root_plot.setContentsMargins(0,0,0,0)

In [ ]:
a_spectogram_widget.set #  .params
# a_spectogram_widget.params.image_bounds_extent
# self.params.x_range
# , padding=0.05

In [ ]:
a_spectogram_widget.layout().contentsMargins() #.getContentsMargins

In [ ]:
# a_spectogram_widget.layout().setContentsMargins(1,1,1,1)
a_spectogram_widget.layout().setContentsMargins(0,0,0,0)
a_spectogram_widget.setLayout(a_spectogram_widget.layout())
a_spectogram_widget.layout().setHorizontalSpacing(0)
a_spectogram_widget.layout().setVerticalSpacing(0)

# Motion Bad Period Detection

### Motion track: BAD intervals (exclude + dark overlay)

`MotionTrackDatasource` accepts optional bad segments aligned with `motion_df['t']` (Unix seconds or datetimes in the same time base as the track).

- **`bad_intervals_df`**: either columns `t_start` + `t_duration`, or MNE-style `onset` + `duration` (then pass **`bad_intervals_time_origin_unix`** = recording start as Unix seconds so `t_start = origin + onset`).
- **`exclude_bad_from_detail`** (default `True`): rows whose time falls inside a bad interval are dropped **before** downsampling.
- **`bad_overlay_alpha`** (default `0.9`): vertical black bands drawn on top of the motion lines via `LinearRegionItem`.

Helpers: `normalize_motion_bad_intervals_df`, `motion_bad_intervals_key_suffix`.

**PhoPyMNEHelper `is_moving_annots_df`** (has `onset`, `duration` in seconds relative to the recording):

```python
from pypho_timeline.rendering.datasources.specific.motion import MotionTrackDatasource, normalize_motion_bad_intervals_df

recording_start_unix = float(your_meas_date.timestamp())  # same origin as timeline motion `t`
ds = MotionTrackDatasource(
    intervals_df, motion_df,
    bad_intervals_df=is_moving_annots_df,
    bad_intervals_time_origin_unix=recording_start_unix,
    exclude_bad_from_detail=True,
    bad_overlay_alpha=0.9,
)
```

If bad intervals are already in absolute Unix time, use `bad_intervals_df=normalize_motion_bad_intervals_df(df)` and omit `bad_intervals_time_origin_unix`.

After **`set_bad_intervals(...)`** on an already displayed track, refresh the cached renderer if needed: `track_renderer.detail_renderer = ds.get_detail_renderer()` (and re-trigger a viewport refresh / overview update as your app does for `source_data_changed_signal`).

In [ ]:
eeg_ds.compute()

In [ ]:
eeg_ds.on_compute_finished()

In [ ]:
eeg_ds

In [ ]:
if getattr(eeg_ds, 'merged_bad_epoch_intervals_plot_callback_fn', None) is not None:
    eeg_ds.merged_bad_epoch_intervals_plot_callback_fn(timeline=timeline)

LiveWindowEventIntervalMonitoringMixin.on_visible_intervals_changed()
LiveWindowEventIntervalMonitoringMixin.on_visible_event_intervals_added(added_rows: {'LOG_TextLogger':         t_start  t_duration         t_end
0  1.777547e+09  224.930717  1.777547e+09, 'EEG_Spectrogram_Epoc X_Frontal':         t_start  t_duration         t_end
0  1.777547e+09   367.05528  1.777547e+09, 'EEG_Epoc X':         t_start  t_duration         t_end
0  1.777547e+09   367.05528  1.777547e+09, 'EEG_Spectrogram_Epoc X_Posterior':         t_start  t_duration         t_end
0  1.777547e+09   367.05528  1.777547e+09, 'MOTION_Epoc X Motion':         t_start  t_duration         t_end
0  1.777547e+09  353.387759  1.777547e+09})
LiveWindowEventIntervalMonitoringMixin.visible_event_intervals_removed(removed_rows: {'LOG_TextLogger':         t_start   t_duration         t_end
4  1.777507e+09     0.000000  1.777507e+09
3  1.777509e+09  2591.146026  1.777512e+09
2  1.777525e+09  1952.424541  1.777527e+09
1  1.777546e+09  

In [ ]:
eeg_ds.eeg_comps_flat_concat_dict

In [ ]:
import phopymnehelper.type_aliases as types

# eeg_ds.extract_all_datasets_results
# type(eeg_ds.computed_result)
# list(eeg_ds.computed_result.keys()) # ['time_independent_bad_channels', 'bad_epochs', 'spectogram']
a_comp_name_key: str = 'spectogram'
a_comp_result = eeg_ds.computed_result[a_comp_name_key]
type(a_comp_result) # list
len(a_comp_result) # 7 == len(n_intervals)

a_sess_idx: int = 0
a_sess_specific_comp_result = a_comp_result[a_sess_idx]
type(a_sess_specific_comp_result)
len(a_sess_specific_comp_result)

list(a_sess_specific_comp_result.keys()) # ['t', 'freqs', 'fs', 'ch_names', 'spectogram_result_dict', 'Sxx_avg', 'Sxx']

In [ ]:
computed_result: Dict[types.EEGComputationId, List[Dict[str, Any]]] = eeg_ds.computed_result # Each list has one entry per eeg_sess

desired_sess_idx: int = (eeg_ds.num_sessions - 1) ## last session
filtered_computed_result: Dict[types.EEGComputationId, Dict[str, Any]] = eeg_ds.get_computed_results_for_sess(sess_idx=desired_sess_idx)
filtered_computed_result


In [ ]:
from phopymnehelper.analysis.computations.eeg_registry import run_eeg_computations_graph, session_fingerprint_for_raw_or_path
from phopymnehelper.analysis.computations.specific.bad_epochs import ensure_bad_epochs_interval_track

eeg_comps_results_dict = {}
eeg_comps_flat_concat_dict: Dict[types.EEGComputationId, Any] = {} ## A dict with keys like {"time_independent_bad_channels": {}, "bad_epochs": {}}

for a_sess_xdf_filename, eeg_raw_list in eeg_ds.raw_datasets_dict.items():
    if eeg_raw_list is not None:
        for eeg_raw in eeg_raw_list:
            if eeg_raw is not None:
                eeg_comps_result = run_eeg_computations_graph(eeg_raw, session=session_fingerprint_for_raw_or_path(eeg_raw), goals=("time_independent_bad_channels", "bad_epochs",))
                if a_sess_xdf_filename not in eeg_comps_results_dict:
                    eeg_comps_results_dict[a_sess_xdf_filename] = []
                eeg_comps_results_dict[a_sess_xdf_filename].append(eeg_comps_result)
                for kk, v in eeg_comps_result.items():
                    if v is not None:
                        if kk not in eeg_comps_flat_concat_dict:
                            eeg_comps_flat_concat_dict[kk] = []
                        eeg_comps_flat_concat_dict[kk].append(v)

                # time_independent_bad_channels = eeg_comps_result["time_independent_bad_channels"]
                # bad_epochs = eeg_comps_result["bad_epochs"]
                # time_independent_bad_channels
                # bad_epochs

                # bad_epoch_intervals_rel = bad_epochs['bad_epoch_intervals_rel']
                # bad_epoch_intervals_df: pd.DataFrame = pd.DataFrame(bad_epoch_intervals_rel, columns=['start_t_rel', 'end_t_rel'])
                # t_col_names: str = ['start_t', 'end_t']
                # for a_t_col in t_col_names:
                #     bad_epoch_intervals_df[a_t_col] = bad_epoch_intervals_df[f'{a_t_col}_rel'] + t0

## OUTPUTS: eeg_comps_flat_concat_dict

In [ ]:
k

In [ ]:
from phopymnehelper.analysis.computations.specific.bad_epochs import ensure_bad_epochs_interval_track

# t0: float = eeg_ds.earliest_unix_timestamp
# ensure_bad_epochs_interval_track(timeline, merged_bad_epoch_intervals_df, time_offset=t0)
ensure_bad_epochs_interval_track(timeline, merged_bad_epoch_intervals_df, time_offset=0)
bad_epochs_widget, bad_epochs_track, bad_epochs_ds = timeline.get_track_tuple("bad epoch intervals")
bad_epochs_layout_plot_item = bad_epochs_widget.getRootPlotItem()

# detailed_eeg_df: pd.DataFrame = bad_epochs_ds.detailed_df

In [ ]:
from phopymnehelper.analysis.computations.specific.bad_epochs import ensure_bad_epochs_interval_track, apply_bad_epochs_overlays_to_timeline

new_regions = apply_bad_epochs_overlays_to_timeline(timeline, merged_bad_epoch_intervals_df, time_offset=0)


In [ ]:
eeg_ds.on_compute_finished()

In [ ]:
eeg_ds.compute()

In [ ]:
eeg_ds.merged_bad_epoch_intervals_plot_callback_fn(eeg_ds, timeline)

## Override downsampling

In [ ]:
# timeline.track_datasources[motion_track.track_id].set_downsampling(max_points_per_second=50.0)

timeline.track_datasources[motion_track.track_id].set_downsampling(max_points_per_second=100) # 100: 10ms, 50: 20ms

In [ ]:
timeline.track_datasources[eeg_track.track_id].set_downsampling(max_points_per_second=100) # 100: 10ms, 50: 20ms

In [ ]:
# motion_track.track_id
motion_ds.max_points_per_second

In [ ]:
## Adjust the downsampling for the EEG 
eeg_ds.max_points_per_second = 50.0   # higher-res than the default 10.0
eeg_ds.enable_downsampling = True     # keep downsampling on, but less aggressive
# Or disable datasource downsampling completely:
# ds.enable_downsampling = False
# ds.max_points_per_second = None

# Important: clear cached detail and force a re-fetch/rerender
track_name: str = eeg_track.track_id
print(f'track_name: {track_name}')
timeline.async_detail_fetcher.cancel_all_pending_fetches(track_name)
timeline.async_detail_fetcher.clear_cache()   # safest option currently
a_renderer = timeline.track_renderers[track_name]
# a_detail_renderer = a_renderer.detail_renderer # MotionPlotDetailRenderer 

a_renderer.clear_all_details()
x0, x1 = a_renderer.plot_item.getViewBox().viewRange()[0]
a_renderer.update_viewport(x0, x1)


# New custom EEG Spectogram

In [ ]:
eeg_ds.compute()

In [ ]:
desired_sess_idx: int = (eeg_ds.num_sessions - 1) ## last session
filtered_computed_result: Dict[types.EEGComputationId, Dict[str, Any]] = eeg_ds.get_computed_results_for_sess(sess_idx=desired_sess_idx)
filtered_computed_result

In [ ]:
eeg_ds.compute()

In [ ]:
from phopymnehelper.EEG_data import EEGComputations, EEGData

_out_all = []
for desired_sess_idx, a_raw in enumerate(eeg_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)):
    _out = EEGComputations.time_independent_bad_channels(raw=a_raw)
    # _out = EEGComputations.apply_autoreject_filter(a_raw=a_raw)
    # _out = EEGComputations.time_dependent_bad_channels(raw=a_raw)
    _out_all.append(_out)


# _out_all
# a_raw = eeg_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)[desired_sess_idx]
# a_raw

In [ ]:
from mne.preprocessing import annotate_movement, annotate_muscle_zscore, annotate_amplitude, annotate_break

a_raw = a_raw.copy()

# Flat channels / segments
flat_annot, flat_chs = annotate_amplitude(a_raw)

# High-frequency muscle noise (proxy for noisy channels)
muscle_annot, scores = annotate_muscle_zscore(a_raw)

a_raw.set_annotations(flat_annot + muscle_annot)

In [ ]:
from pyprep.prep_pipeline import PrepPipeline
from mne.channels.montage import DigMontage
from phopymnehelper.anatomy_and_electrodes import ElectrodeHelper

active_electrode_man: ElectrodeHelper = ElectrodeHelper.init_EpocX_montage()
emotiv_epocX_montage: DigMontage = active_electrode_man.active_montage

a_raw = a_raw.copy()
a_raw.set_montage(emotiv_epocX_montage)

prep = PrepPipeline(
    a_raw.copy(),
    prep_params={
        "ref_chs": "eeg",
        "reref_chs": "eeg",
        "line_freqs": [60],
    },
    montage=emotiv_epocX_montage,  # or your montage
    channel_wise=True,
)

prep.fit()
prep


In [ ]:
bad_chs = prep.interpolated_channels_
bad_chs


In [ ]:
from mne_faster import (
    find_bad_channels,
    find_bad_channels_in_epochs,
    find_bad_components,
    find_bad_epochs,
)

###############################################################################
# Clean the data using FASTER

# Step 1: mark bad channels
epochs.info["bads"] = find_bad_channels(epochs, eeg_ref_corr=True)
if len(epochs.info["bads"]) > 0:
    epochs.interpolate_bads()



In [ ]:
from phopymnehelper.EEG_data import EEGComputations, EEGData

_out = EEGComputations.time_independent_bad_channels(raw=a_raw)
_out

In [ ]:
_all_outputs = EEGComputations.run_all(raw=raw)



In [ ]:

'time_independent_bad_channels'
# filtered_computed_result['time_independent_bad_channels']

In [ ]:
timeline.active_window_visible_intervals_dict
(timeline.active_window_start_time, timeline.active_window_end_time)

_found_intervals = timeline.find_intervals_in_active_window(debug_print=True)
_found_intervals

In [ ]:
# eeg_overview_intervals_df
eeg_detail_renderer = eeg_ds.get_detail_renderer()
eeg_detail_renderer

In [ ]:
eeg_ds.intervals_df

In [ ]:
timeline.total_data_start_time

In [ ]:
eeg_ds.raw_datasets_dict

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from phopymnehelper.analysis.computations.specific.EEG_Spectograms import compute_raw_eeg_spectrogram

# raw = your_mne_raw  # mne.io.Raw with EEG

for k, raw in eeg_ds.raw_datasets_dict.items():

    spec = compute_raw_eeg_spectrogram(
        raw,
        nperseg=2048,
        noverlap=512,
        picks=None,
        mask_bad_annotated_times=True,
    )

    freqs, t = np.asarray(spec["freqs"]), np.asarray(spec["t"])
    Sxx = np.asarray(spec["Sxx"])
    Z = 10.0 * np.log10(np.nanmean(Sxx, axis=0) + 1e-12)
    fmin, fmax = 1.0, 40.0
    m = (freqs >= fmin) & (freqs <= fmax)

    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.pcolormesh(t, freqs[m], Z[m, :], shading="auto", cmap="viridis")
    ax.set_ylim(fmin, fmax)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title("EEG spectrogram (mean over channels)")
    fig.colorbar(im, ax=ax, label="10·log10(power)")
    plt.tight_layout()
    plt.show()



## Explore

In [ ]:
# new_brush = pg.mkBrush(0, 0, 0, int(round(255 * 0.9)))
# new_brush = pg.mkBrush(255, 255, 10, int(round(255 * 0.4)))
new_brush = pg.mkBrush(255, 255, 10, int(round(255 * 0.7)))
new_pen = pg.mkPen(None)

for a_track_id, a_regions_list in new_regions.items():
    # a_regions_list[0].setBrush(pg.mkBrush(0, 0, 0, 0.25))
    for a_region in a_regions_list:
        a_region.setBrush(new_brush)
        # a_region.setVisible(True)

In [ ]:
bad_epochs_widget.plots
bad_epochs_widget.plots_data

In [ ]:
# raws = RawProvidingTrackDatasource.get_sorted_and_extracted_raws(raw_datasets_dict)

raws = eeg_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)
raws


In [ ]:
raws = motion_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)
raws

In [ ]:

# rep, lst = compute_multiraw_spectrogram_results(eeg_ds.intervals_df, eeg_ds.raw_datasets_dict)


In [ ]:
# bad_epochs_intervals_df_t_col_names: str = ['t_start', 't_end']

# merged_bad_epoch_intervals_df = merged_bad_epoch_intervals_df.rename(columns=dict(zip(['t_start', 't_end', 't_start_rel', 't_end_rel', 't_start_dt', 't_end_dt'], ['start_t', 'end_t', 'start_t_rel', 'end_t_rel', 'start_t_dt', 'end_t_dt'])), inplace=False)
# merged_bad_epoch_intervals_df = merged_bad_epoch_intervals_df.rename(columns=dict(zip(['start_t', 'end_t', 'start_t_rel', 'end_t_rel', 'start_t_dt', 'end_t_dt'], ['t_start', 't_end', 't_start_rel', 't_end_rel', 't_start_dt', 't_end_dt'])), inplace=False)
merged_bad_epoch_intervals_df[bad_epochs_intervals_df_t_col_names] -= earliest_interval_t_start
merged_bad_epoch_intervals_df


In [ ]:
merged_bad_epoch_intervals_df[bad_epochs_intervals_df_t_col_names]

In [ ]:
(merged_bad_epoch_intervals_df[bad_epochs_intervals_df_t_col_names] - t0)

In [ ]:
ds_overview_intervals_df['t_start'].to_numpy() - np.nanmin(ds_overview_intervals_df['t_start'].to_numpy())

In [ ]:
ds_overview_intervals_df['t_start'].to_numpy() - t0

In [ ]:
earliest_interval_t_start: float = np.nanmin(ds_overview_intervals_df['t_start'].to_numpy())
(merged_bad_epoch_intervals_df[bad_epochs_intervals_df_t_col_names] - earliest_interval_t_start)


(merged_bad_epoch_intervals_df[bad_epochs_intervals_df_t_col_names] - np.nanmin(ds_overview_intervals_df['t_start'].to_numpy()))

In [ ]:
# np.all(merged_bad_epoch_intervals_df[bad_epochs_intervals_df_t_col_names] >= t0)
merged_bad_epoch_intervals_df[bad_epochs_intervals_df_t_col_names] >= earliest_interval_t_start



In [ ]:
merged_bad_epoch_intervals_df

In [ ]:
from phopymnehelper.analysis.computations.specific.bad_epochs import ensure_bad_epochs_interval_track

# t0: float = eeg_ds.earliest_unix_timestamp
# ensure_bad_epochs_interval_track(timeline, merged_bad_epoch_intervals_df, time_offset=t0)
ensure_bad_epochs_interval_track(timeline, merged_bad_epoch_intervals_df, time_offset=0)

## overflow

In [ ]:
bad_epoch_intervals_rel = bad_epochs['bad_epoch_intervals_rel']
bad_epoch_intervals_df: pd.DataFrame = pd.DataFrame(bad_epoch_intervals_rel, columns=['start_t_rel', 'end_t_rel'])
t_col_names: str = ['start_t', 'end_t']
for a_t_col in t_col_names:
    bad_epoch_intervals_df[a_t_col] = bad_epoch_intervals_df[f'{a_t_col}_rel'] + t0

bad_epoch_intervals_df.diff().max()




In [ ]:
eeg_ds.detailed_df['t']
eeg_ds.intervals_df

In [ ]:
eeg_raw.first_time
eeg_raw.times
eeg_raw.info.meas_data

In [ ]:
eeg_comps_flat_concat_dict['bad_epochs']

In [ ]:
[v.get('bad_epoch_intervals_df', None) for v in eeg_comps_flat_concat_dict['bad_epochs']]
# pd.concat([v.get('bad_epoch_intervals_df', None) for v in eeg_comps_flat_concat_dict['bad_epochs']])



# eeg_comps_results_dict

In [ ]:
t0: float = eeg_ds.earliest_unix_timestamp
t0


In [ ]:
# eeg_ds.detailed_df
eeg_ds.intervals_df

In [ ]:
import pandas as pd
import numpy as np

eeg_df = eeg_ds.detailed_df.sort_values("t").reset_index(drop=True)
_t = eeg_df["t"]
if pd.api.types.is_datetime64_any_dtype(_t):
    t0 = float(pd.to_datetime(_t, utc=True, errors="coerce").astype(np.int64).iloc[0] / 1e9)
else:
    t0 = float(pd.to_numeric(_t, errors="coerce").iloc[0])

t0

In [ ]:
eeg_raw.

In [ ]:
timeline.get_all_track_names()

In [ ]:
eeg_spectogram_track_names = ['EEG_Spectrogram_Epoc X_Frontal-L',
 'EEG_Spectrogram_Epoc X_Frontal-R',
 'EEG_Spectrogram_Epoc X_Posterior-L',
 'EEG_Spectrogram_Epoc X_Posterior-R',]

for a_spectogram_track_name in eeg_spectogram_track_names:
    a_spectogram_widget, a_spectogram_track, a_spectogram_ds = timeline.get_track_tuple(a_spectogram_track_name)
    # a_spectogram_track
    a_root_graphics_layout_widget = a_spectogram_widget.getRootGraphicsLayoutWidget()
    a_plot_item = a_spectogram_widget.getRootPlotItem()
    a_plot_item.hideAxis('bottom')
    # a_spectogram_ds.detailed_df

In [ ]:
## How can I mask an existing track (such as the Motion Track) using a set of BAD_INTERVALS like is_moving_annots_df: pd.DataFrame? I'd like the data object to be set to indicate that these periods should be excluded, and the BAD periods to be rendered darkened by overlaying a BLACK fill with 90% opacity for the regions
# motion_track.set_bad_intervals(bad_intervals_df=is_moving_annots_df)
motion_ds.set_bad_intervals(bad_intervals_df=is_moving_annots_df)

## End motion annotations

### Timeline timewindow hacking

In [ ]:
timeline.go_to_latest_window() # timeline.go_to_latest_window()

In [ ]:
## this seems wrong `timeline.total_data_end_time`
(timeline.total_data_start_time, timeline.total_data_end_time) ## (datetime.datetime(2026, 4, 16, 7, 14, 0, 23324, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 16, 16, 5, 55, 326405, tzinfo=datetime.timezone.utc))
(timeline.active_window_start_time, timeline.active_window_end_time) # (datetime.datetime(2026, 4, 16, 7, 14, 0, 23324, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 16, 22, 57, 5, 323324, tzinfo=datetime.timezone.utc))
(timeline._last_applied_plot_window_x0, timeline._last_applied_plot_window_x1) # (1776323640.023324, 1776380225.323324)

In [ ]:
timeline.spikes_window.active_time_window

In [ ]:
timeline.spikes_window.update_window_start_end(timeline.active_window_start_time, timeline.active_window_end_time)


# Compute Spectograms

In [ ]:
from pypho_timeline.rendering.datasources.specific.eeg import EEGPlotDetailRenderer, EEGTrackDatasource, SpectrogramChannelGroupConfig, compute_multiraw_spectrogram_results


spect_groups = [SpectrogramChannelGroupConfig(name='All', channels=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']),
    SpectrogramChannelGroupConfig(name='Frontal', channels=['AF3', 'F7', 'FC5', 'F3', 'AF4', 'F8', 'FC6', 'F4']),
    SpectrogramChannelGroupConfig(name='Posterior', channels=['T7', 'P7', 'O1', 'T8', 'P8', 'O2']),
    # SpectrogramChannelGroupConfig(name='Left', channels=['AF3', 'F7', 'FC5', 'F3', 'T7', 'P7', 'O1']),
]

_out_spect_ds_list = eeg_ds.add_spectrogram_tracks_for_channel_groups(spectrogram_channel_groups=spect_groups, timeline=timeline, timeline_builder=builder)



In [8]:
timeline.hide_extra_xaxis_labels_and_axes(enable_hide_extra_track_x_axes=True)

In [ ]:
spec_names = [n for n in timeline.track_datasources if n.startswith("EEG_Spectrogram_")]
spec_names += [n for n in timeline.track_datasources if n.startswith("EEG_Spectrogram_") and n.endswith("__compare")]
spec_names

In [ ]:
# After: timeline = builder.build_from_xdf_files(...)
assert timeline is not None and len(timeline.get_all_track_names()) > 0, "Build the timeline first."
_names = timeline.get_all_track_names()
_eeg_names = [n for n in _names if n.startswith("EEG_")]
_motion_names = [n for n in _names if n.startswith("MOTION_")]
# if not _eeg_names:
#     raise RuntimeError(f"No EEG_* track found. Names: {_names}")
# eeg_name = _eeg_names[0]
eeg_name = 'EEG_Epoc X' ## HARDCODED
print(f'eeg_name: {eeg_name}')
motion_name = _motion_names[0] if _motion_names else None
eeg_ds = timeline.track_datasources[eeg_name]
eeg_df = eeg_ds.detailed_df.sort_values("t").reset_index(drop=True).copy()
_t_ts = eeg_df["t"]
if pd.api.types.is_datetime64_any_dtype(_t_ts):
    eeg_df["t"] = pd.to_datetime(_t_ts, utc=True, errors="coerce").astype(np.int64) / 1e9
else:
    eeg_df["t"] = pd.to_numeric(_t_ts, errors="coerce")

# # eeg_ds.intervals_df
# from pypho_timeline.rendering.datasources.specific.eeg import _first_nonempty_raw_list_from_dict
# _eeg_raws = _first_nonempty_raw_list_from_dict(eeg_ds.raw_datasets_dict)
# assert len(_eeg_raws) > 0, f"eeg datasource has no raw datasets!"
# eeg_raw = _eeg_raws[0]
# eeg_raw

In [ ]:
eeg_ds

In [ ]:
eeg_raw.annotations

In [ ]:
spec_result = eeg_comps_result["spectogram"]

spec_result


In [ ]:
eeg_ds.compute()

In [ ]:
from pypho_timeline.rendering.datasources.specific.eeg import EEGSpectrogramTrackDatasource
from pypho_timeline.rendering.datasources.stream_to_datasources import default_dock_named_color_scheme_key
from pypho_timeline.docking.dock_display_configs import CustomCyclicColorsDockDisplayConfig, NamedColorScheme
from pypho_timeline.core.synchronized_plot_mode import SynchronizedPlotMode
from qtpy import QtWidgets

track_name = "EEG_Spectrogram_all"  # unique
spec_ds = EEGSpectrogramTrackDatasource(
    intervals_df=eeg_ds.intervals_df.copy(),
    spectrogram_result=spec_result,
    custom_datasource_name=track_name,
    group_config=None,
    lab_obj_dict=eeg_ds.lab_obj_dict,
    raw_datasets_dict=eeg_ds.raw_datasets_dict,
    parent=eeg_ds,
)

_scheme_key = default_dock_named_color_scheme_key(track_name)
display_config = CustomCyclicColorsDockDisplayConfig(
    named_color_scheme=NamedColorScheme[_scheme_key],
    showCloseButton=True, showCollapseButton=True, showGroupButton=True, corner_radius="3px",
)
track_widget, root_g, plot_item, dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
    name=track_name,
    dockSize=(500, 80),
    dockAddLocationOpts=["bottom"],
    display_config=display_config,
    sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA,
)
# same x-axis setup as TimelineBuilder._add_tracks_to_timeline …
plot_item.setYRange(0, 1, padding=0)
plot_item.setLabel("left", track_name)
plot_item.hideAxis("left")
timeline.add_track(spec_ds, name=track_name, plot_item=plot_item)
track_widget.optionsPanel = track_widget.getOptionsPanel()
if hasattr(dock, "updateWidgetsHaveOptionsPanel"):
    dock.updateWidgetsHaveOptionsPanel()
getattr(timeline, "_rebuild_timeline_overview_strip", lambda: None)()

In [ ]:
spec_ds.raw_datasets_dict

## ADHD / theta–delta sleep intrusion (`compute_theta_delta_sleep_intrusion_series`)

Run **after** the timeline is built (previous cell). Resolves `EEG_*` and optional `MOTION_*` tracks, builds `mne.io.RawArray` from full-rate `detailed_df`, aligns motion times to `raw.times`, and stashes `adhd_ctx` for the next cells.

In [ ]:
from phopymnehelper.analysis.computations.eeg_registry import run_eeg_computations_graph, session_fingerprint_for_raw_or_path
from phopymnehelper.analysis.computations.specific.ADHD_sleep_intrusions import apply_adhd_sleep_intrusion_to_timeline
from phopylslhelper.datetime_helpers import unix_timestamp_to_datetime, datetime_to_unix_timestamp
from datetime import datetime, timedelta

eeg_name = timeline.eeg_track_identifier            # or 'EEG_Epoc X'
eeg_ds   = timeline.track_datasources[eeg_name]
eeg_raw  = eeg_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)[0]

result = run_eeg_computations_graph(eeg_raw, session=session_fingerprint_for_raw_or_path(eeg_raw), goals=("theta_delta_sleep_intrusion",))["theta_delta_sleep_intrusion"]
print("session_mean:", result["session_mean_theta_delta"], "valid:", result["n_valid_windows"], "/", result["n_windows"])

td_ratio_widget, td_ratio_track, td_ratio_ds = apply_adhd_sleep_intrusion_to_timeline(timeline, result, eeg_name=eeg_name, eeg_ds=eeg_ds)
detailed_td_ratio_df: pd.DataFrame = td_ratio_ds.detailed_df

In [ ]:
td_ratio_widget.active_plot_target.viewRange() # [[1777458500.7219324, 1777460685.571476], [-0.02, 1.02]]

np.diff([1777458500.7219324, 1777460685.571476])


In [ ]:
x0v, x1v = ref_plot.getViewBox().viewRange()[0]
ratio_plot_item.setXRange(x0v, x1v, padding=0)

ratio_plot_item.setYRange(0, 1, padding=-0.02)

In [ ]:
td_ratio_widget, td_ratio_track, td_ratio_ds = timeline.get_track_tuple("ANALYSIS_theta_delta")
detailed_td_ratio_df: pd.DataFrame = td_ratio_ds.detailed_df
detailed_td_ratio_df

In [ ]:
detailed_td_ratio_df['dt'] = unix_timestamp_to_datetime(detailed_td_ratio_df['t'].to_numpy())
detailed_td_ratio_df

In [ ]:
timeline.reference_datetime

In [ ]:

eeg_ds.reference_datetime

In [ ]:
t0_unix = result.get("t0_unix")
t0_earliest_unix_ts = float(eeg_ds.earliest_unix_timestamp)
t0_earliest_unix_ts
t0_unix - t0_earliest_unix_ts ## earliest unix_timestamp is actually later!
dt0_unix = unix_timestamp_to_datetime(t0_unix)
dt0_earliest_unix_ts = unix_timestamp_to_datetime(t0_earliest_unix_ts)
## OUTPUTS: dt0_unix, dt0_unix
dt0_unix, dt0_earliest_unix_ts

# (datetime.datetime(2026, 4, 27, 23, 42, 28, tzinfo=datetime.timezone.utc),
#  datetime.datetime(2026, 4, 29, 10, 38, 46, 298468, tzinfo=datetime.timezone.utc))
eeg_track_correction_delta: timedelta = (dt0_unix - dt0_earliest_unix_ts) # datetime.timedelta(days=-2, seconds=47021, microseconds=701532)

## find mice, call mice, mice are nice
eeg_track_correction_delta

## datetime.timedelta(days=-2, seconds=56444, microseconds=448690) - original offset
## datetime.timedelta(days=-1, seconds=64124, microseconds=812851) - Adding another session reduced the time, I guess because this knocked off one of the earliest sessions due to the fixed num active sessions

## I guess I lost the memories due to not taking the Mitrapizine last night. I also didn't take it for some days before. How sad. 



# Use the `eeg_track_correction_delta` to correct all the EEG-derived tracks:


In [ ]:
## Use the `eeg_track_correction_delta` to correct all the EEG-derived tracks:
eeg_track_correction_delta

eeg_ds.earliest_unix_timestamp


In [ ]:
## apply it to all tracks in the timeline


In [ ]:
import datetime
import pandas as pd
from qtpy import QtCore

eeg_track_correction_delta = datetime.timedelta(days=-2, seconds=78719, microseconds=635839)
delta_seconds: float = eeg_track_correction_delta.total_seconds()  # ≈ -94080.36

INTERVAL_DF_TIME_COLS = ('t_start', 't_end')
DETAILED_DF_TIME_COLS = ('t',)
EXTRA_INTERVAL_TIME_COLS = ('t_start_dt', 't_end_dt', 't_start_timeline_rel', 't_end_timeline_rel')

def _shift_df_columns(df, cols, delta_seconds):
    """Shift any present time columns in `df` by `delta_seconds` (handles datetime + numeric)."""
    if df is None:
        return
    for col in cols:
        if col not in df.columns:
            continue
        s = df[col]
        if pd.api.types.is_datetime64_any_dtype(s):
            df[col] = s + pd.Timedelta(seconds=delta_seconds)
        elif pd.api.types.is_timedelta64_dtype(s):
            continue  # durations don't shift
        else:
            df[col] = s + delta_seconds

def shift_track_datasource_in_place(ds, delta_seconds):
    """Shift every time-bearing dataframe known to this datasource. Safe across EEG/Motion/Video/Dose/Interval."""
    _shift_df_columns(getattr(ds, 'intervals_df', None), INTERVAL_DF_TIME_COLS + EXTRA_INTERVAL_TIME_COLS, delta_seconds)
    _shift_df_columns(getattr(ds, 'detailed_df', None), DETAILED_DF_TIME_COLS, delta_seconds)
    nref = getattr(ds, 'normalization_reference_df', None)
    detailed = getattr(ds, 'detailed_df', None)
    if nref is not None and nref is not detailed:
        _shift_df_columns(nref, DETAILED_DF_TIME_COLS, delta_seconds)
    bad_eps = getattr(ds, 'merged_bad_epoch_intervals_df', None)
    if bad_eps is not None:
        _shift_df_columns(bad_eps, INTERVAL_DF_TIME_COLS, delta_seconds)
    df_attr = getattr(ds, '_df', None)
    if df_attr is not None and df_attr is not getattr(ds, 'intervals_df', None):
        _shift_df_columns(df_attr, INTERVAL_DF_TIME_COLS, delta_seconds)

if getattr(timeline, '_corrected_for_eeg_drift', False):
    raise RuntimeError("Timeline already shifted once this session. Re-run only if you reset state.")

track_names = list(timeline.track_datasources.dynamically_added_attributes)
interval_names = list(getattr(timeline, 'interval_datasource_names', []) or [])

print(f'Shifting {len(track_names)} track datasources and {len(interval_names)} interval datasources by {delta_seconds:+.3f} s')

for name in track_names:
    ds = timeline.track_datasources.get(name, None)
    if ds is None:
        continue
    was_blocked = ds.blockSignals(True)
    try:
        shift_track_datasource_in_place(ds, delta_seconds)
    finally:
        ds.blockSignals(was_blocked)

for name in interval_names:
    ds = timeline.interval_datasources.get(name, None)
    if ds is None:
        continue
    was_blocked = ds.blockSignals(True)
    try:
        df_obj = getattr(ds, 'df', None)
        _shift_df_columns(df_obj, INTERVAL_DF_TIME_COLS + EXTRA_INTERVAL_TIME_COLS, delta_seconds)
    finally:
        ds.blockSignals(was_blocked)

def _refresh_all_renderers():
    for tname, tr in list(timeline.track_renderers.items()):
        on_changed = getattr(tr, '_on_datasource_changed', None)
        if on_changed is not None:
            try:
                on_changed()
            except Exception as e:
                print(f"  [warn] renderer refresh failed for {tname!r}: {e}")
    for tname, tr in list(timeline.track_renderers.items()):
        plot_item = getattr(tr, 'plot_item', None)
        if plot_item is None:
            continue
        vb = plot_item.getViewBox()
        if vb is None:
            continue
        (x0, x1), _ = vb.viewRange()
        vb.setXRange(x0 + delta_seconds, x1 + delta_seconds, padding=0)

QtCore.QTimer.singleShot(0, _refresh_all_renderers)

timeline._corrected_for_eeg_drift = True
print('Correction queued; renderers will refresh on the next event-loop tick.')


In [9]:
[timeline.add_new_now_line_for_plot_item(widget.getRootPlotItem()) for widget in timeline.ui.matplotlib_view_widgets.values()]
timeline.update_now_lines()

In [ ]:
## update the EEG track with the correction to see if it shows the real datetimes after that.
import datetime
import pandas as pd
from pypho_timeline.rendering.datasources.specific.eeg import EEGTrackDatasource

eeg_track_correction_delta = datetime.timedelta(days=-2, seconds=78719, microseconds=635839)
delta_seconds = eeg_track_correction_delta.total_seconds()

eeg_name = 'EEG_Epoc X'
old_ds: EEGTrackDatasource = timeline.track_datasources[eeg_name]

eeg_widget, _, _ = timeline.get_track_tuple(eeg_name)
eeg_plot = eeg_widget.getRootPlotItem() if eeg_widget is not None else None

new_intervals_df = old_ds.intervals_df.copy()
new_intervals_df['t_start'] = new_intervals_df['t_start'] + delta_seconds
if 't_end' in new_intervals_df.columns:
    new_intervals_df['t_end'] = new_intervals_df['t_end'] + delta_seconds

new_detailed_df = old_ds.detailed_df.copy() if old_ds.detailed_df is not None else None
if new_detailed_df is not None and 't' in new_detailed_df.columns:
    new_detailed_df['t'] = new_detailed_df['t'] + delta_seconds

timeline.remove_track(eeg_name)

new_ds = EEGTrackDatasource(intervals_df=new_intervals_df, eeg_df=new_detailed_df, custom_datasource_name=eeg_name, max_points_per_second=old_ds.max_points_per_second, enable_downsampling=old_ds.enable_downsampling, fallback_normalization_mode=old_ds.fallback_normalization_mode, normalization_mode_dict=old_ds.normalization_mode_dict, arbitrary_bounds=old_ds.arbitrary_bounds, normalize=old_ds.normalize, normalize_over_full_data=old_ds.normalize_over_full_data, channel_names=old_ds.channel_names, lab_obj_dict=getattr(old_ds, 'lab_obj_dict', None), raw_datasets_dict=getattr(old_ds, 'raw_datasets_dict', None), plot_pen_colors=old_ds.plot_pen_colors, plot_pen_width=old_ds.plot_pen_width)

timeline.add_track(new_ds, name=eeg_name, plot_item=eeg_plot)

In [ ]:
ratio_plot_item.setXRange(x0v, x1v, padding=0)
ratio_plot_item.setYRange(0, 1, padding=-0.02)



In [ ]:
float_to_datetime(detailed_td_ratio_df['t'].to_numpy(), reference_datetime=timeline.reference_datetime)


In [ ]:
td_ratio_track

In [ ]:
result

In [ ]:
from datetime import datetime, timezone

all_raws = eeg_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)
visible_t_start = float(eeg_ds.intervals_df["t_start"].min())
visible_t_end   = float(eeg_ds.intervals_df["t_start"].max() + eeg_ds.intervals_df["t_duration"].max())
print(f"Visible range: {datetime.fromtimestamp(visible_t_start, tz=timezone.utc)} -> {datetime.fromtimestamp(visible_t_end, tz=timezone.utc)}")

for i, r in enumerate(all_raws):
    md = r.info.get("meas_date")
    ts = md.timestamp() if md is not None else None
    in_range = (ts is not None) and (visible_t_start - 60 <= ts <= visible_t_end + 60)
    marker = "  <-- VISIBLE" if in_range else ""
    print(f"[{i}] meas_date={md} unix={ts} duration={r.times[-1]:.1f}s{marker}")

# ==================================================================================================================================================================================================================================================================================== #
# pre 2026-04-28 adjustment:                                                                                                                                                                                                                                                           #
# ==================================================================================================================================================================================================================================================================================== #
# Visible range: 2026-04-29 10:38:46.298468+00:00 -> 2026-04-30 01:19:33.430168+00:00
# [0] meas_date=2026-04-27 23:42:28+00:00 unix=1777333348.0 duration=908.0s
# [1] meas_date=2026-04-27 23:42:28+00:00 unix=1777333348.0 duration=908.0s
# [2] meas_date=2026-04-28 06:21:33+00:00 unix=1777357293.0 duration=52.6s
# [3] meas_date=2026-04-28 06:24:08+00:00 unix=1777357448.0 duration=4660.6s


In [ ]:
from datetime import datetime, timezone
import mne
import numpy as np
import pandas as pd
from phopymnehelper.analysis.computations.specific import compute_theta_delta_sleep_intrusion_series
from phopymnehelper.analysis.computations.specific.ADHD_sleep_intrusions import apply_adhd_sleep_intrusion_to_timeline

#     _motion_names = [n for n in _names if n.startswith("MOTION_")]
#     # if not _eeg_names:
#     #     raise RuntimeError(f"No EEG_* track found. Names: {_names}")
#     # eeg_name = _eeg_names[0]
#     eeg_name = 'EEG_Epoc X' ## HARDCODED
#     print(f'eeg_name: {eeg_name}')
#     motion_name = _motion_names[0] if _motion_names else None

#     motion_name = timeline.motion_track_identifier

#     eeg_name = timeline.eeg_track_identifier
#     eeg_ds = timeline.track_datasources[eeg_name]
#     eeg_df = eeg_ds.detailed_df.sort_values("t").reset_index(drop=True).copy()


# # ['LOG_TextLogger',
# #  'MOTION_Epoc X Motion',
# #  'EEG_Spectrogram_Epoc X_Frontal-L',
# #  'EEG_Spectrogram_Epoc X_Frontal-R',
# #  'EEG_Spectrogram_Epoc X_Posterior-L',
# #  'EEG_Spectrogram_Epoc X_Posterior-R',
# #  'EEG_Epoc X']


# # After: timeline = builder.build_from_xdf_files(...)
# assert timeline is not None and len(timeline.get_all_track_names()) > 0, "Build the timeline first."
# _names = timeline.get_all_track_names()
# _eeg_names = [n for n in _names if n.startswith("EEG_")]
# _motion_names = [n for n in _names if n.startswith("MOTION_")]
# # if not _eeg_names:
# #     raise RuntimeError(f"No EEG_* track found. Names: {_names}")
# # eeg_name = _eeg_names[0]
# eeg_name = 'EEG_Epoc X' ## HARDCODED
# print(f'eeg_name: {eeg_name}')
# motion_name = _motion_names[0] if _motion_names else None

# eeg_ds = timeline.track_datasources[eeg_name]
# eeg_df = eeg_ds.detailed_df.sort_values("t").reset_index(drop=True).copy()
# _t_ts = eeg_df["t"]
# if pd.api.types.is_datetime64_any_dtype(_t_ts):
#     eeg_df["t"] = pd.to_datetime(_t_ts, utc=True, errors="coerce").astype(np.int64) / 1e9
# else:
#     eeg_df["t"] = pd.to_numeric(_t_ts, errors="coerce")
# t0 = float(eeg_df["t"].iloc[0])
# _ch = list(eeg_ds.channel_names) if getattr(eeg_ds, "channel_names", None) else [c for c in eeg_df.columns if c != "t"]
# _ch = [c for c in _ch if c in eeg_df.columns]
# _dt = float(np.median(np.diff(eeg_df["t"].to_numpy(dtype=float))))
# sfreq = (1.0 / _dt) if _dt > 0 else 256.0
# assume_microvolts = True
# _data = eeg_df[_ch].to_numpy(dtype=float).T
# if assume_microvolts:
#     _data = _data * 1e-6
# raw_adhd = mne.io.RawArray(_data, mne.create_info(ch_names=list(_ch), sfreq=float(sfreq), ch_types="eeg"))
# raw_adhd.set_meas_date(datetime.fromtimestamp(t0, tz=timezone.utc))
# motion_df_adhd = None
# if motion_name is not None:
#     _mdf = timeline.track_datasources[motion_name].detailed_df.sort_values("t").copy()
#     _mt = _mdf["t"]
#     if pd.api.types.is_datetime64_any_dtype(_mt):
#         _mdf["t"] = pd.to_datetime(_mt, utc=True, errors="coerce").astype(np.int64) / 1e9
#     else:
#         _mdf["t"] = pd.to_numeric(_mt, errors="coerce")
#     _mdf["t"] = _mdf["t"].to_numpy(dtype=float) - t0
#     motion_df_adhd = _mdf
# adhd_ctx = dict(eeg_name=eeg_name, motion_name=motion_name, eeg_ds=eeg_ds, eeg_df=eeg_df, t0=t0, raw=raw_adhd, motion_df=motion_df_adhd, out=None)
# print("eeg_name=", eeg_name, "motion_name=", motion_name, "sfreq=", sfreq, "ch=", len(_ch))

In [ ]:
from phopymnehelper.analysis.computations.eeg_registry import run_eeg_computations_graph, session_fingerprint_for_raw_or_path
from phopymnehelper.analysis.computations.specific.ADHD_sleep_intrusions import ThetaDeltaSleepIntrusionComputation, apply_adhd_sleep_intrusion_to_timeline

active_nearest_sess_idx: int = 2
eeg_raw = eeg_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)[active_nearest_sess_idx]

_curr_compute_key: str = "theta_delta_sleep_intrusion"
eeg_comps_result = run_eeg_computations_graph(eeg_raw, session=session_fingerprint_for_raw_or_path(eeg_raw), goals=(_curr_compute_key,))
_curr_compute_result = eeg_comps_result[_curr_compute_key]
adhd_ctx = _curr_compute_result.copy()
out_adhd = adhd_ctx['out']
print("session_mean_theta_delta", out_adhd["session_mean_theta_delta"], "valid", out_adhd["n_valid_windows"], "/", out_adhd["n_windows"])
print(f'theta_delta_ratio: {out_adhd["theta_delta_ratio"]}')

## OUTPUTS: adhd_ctx, out_adhd

# out_adhd


In [ ]:
adhd_ctx

In [ ]:
out_adhd

In [ ]:
# self.apply_adhd_sleep_intrusion_to_timeline_plot_callback_fn(timeline=timeline)
# out_adhd['apply_adhd_sleep_intrusion_to_timeline_plot_callback_fn'](timeline)

result = eeg_comps_result["theta_delta_sleep_intrusion"]
print("session_mean_theta_delta", result["session_mean_theta_delta"], "valid", result["n_valid_windows"], "/", result["n_windows"])
apply_adhd_sleep_intrusion_to_timeline(timeline, result, eeg_name=eeg_name, eeg_ds=eeg_ds)


In [ ]:
# out_adhd = _curr_compute_result.copy()
# print("session_mean_theta_delta", out_adhd["session_mean_theta_delta"], "valid", out_adhd["n_valid_windows"], "/", out_adhd["n_windows"])
# print(f'theta_delta_ratio: {out_adhd["theta_delta_ratio"]}')

In [ ]:
# Synchronous run (blocks the kernel until done). For GUI threading see the next cell.
out_adhd = compute_theta_delta_sleep_intrusion_series(adhd_ctx["raw"], motion_df=adhd_ctx.get("motion_df", None), window_sec=4.0, step_sec=1.0)
adhd_ctx["out"] = out_adhd
print("session_mean_theta_delta", out_adhd["session_mean_theta_delta"], "valid", out_adhd["n_valid_windows"], "/", out_adhd["n_windows"])


In [ ]:
adhd_ctx["t0"]

In [ ]:
import matplotlib.pyplot as plt
_x = adhd_ctx["t0"] + out_adhd["times"]
plt.figure(figsize=(10, 2.2))
plt.plot(_x, out_adhd["theta_delta_ratio"], color="tab:purple", linewidth=0.8)
plt.xlabel("Time (unix s)")
plt.ylabel("theta / delta")
plt.title("ADHD sleep intrusion series (NaN = motion/QC excluded)")
plt.tight_layout()
plt.show()

In [ ]:
apply_adhd_sleep_intrusion_to_timeline(timeline, adhd_ctx)

# Optional: run compute off the GUI thread, then marshal UI updates to the Qt main thread:
# def run_adhd_in_background():
#     from phopymnehelper.analysis.computations.specific import compute_theta_delta_sleep_intrusion_series
#     fut = ThreadPoolExecutor(max_workers=1).submit(lambda: compute_theta_delta_sleep_intrusion_series(adhd_ctx["raw"], motion_df=adhd_ctx["motion_df"], window_sec=4.0, step_sec=1.0))
#     def _done(f):
#         try:
#             adhd_ctx["out"] = f.result()
#         except Exception as exc:
#             print("ADHD compute failed:", exc)
#             return
#         QtCore.QTimer.singleShot(0, lambda: apply_adhd_sleep_intrusion_to_timeline(timeline, adhd_ctx))
#     fut.add_done_callback(_done)
# run_adhd_in_background()

In [ ]:
from phopymnehelper.analysis.computations.specific.bad_epochs import compute_bad_epochs_qc, apply_bad_epochs_overlays_to_timeline

out = compute_bad_epochs_qc(adhd_ctx["raw"], use_autoreject=True, copy_raw=True)
apply_bad_epochs_overlays_to_timeline(timeline, out, time_offset=t0)

In [ ]:
timeline.hide_extra_xaxis_labels_and_axes()

In [ ]:
timeline.get_all_track_names()


## Futher Timeline Widget/UI Customization

In [10]:
from pypho_timeline.widgets.timeline_overview_strip import TimelineOverviewStrip
from pypho_timeline.EXTERNAL.pyqtgraph_extensions.graphicsObjects.CustomLinearRegionItem import CustomLinearRegionItem

strip: TimelineOverviewStrip = timeline.ui.timeline_overview_strip
region: CustomLinearRegionItem = strip._viewport_region
# strip._intervals_item.setEnabled(True)
# strip._intervals_item.movable = True
# region.setMovable(True) #.movable

In [ ]:
# strip

strip_active_window_float_ts = region.getRegion() # (1776323640.023324, 1776370487.5812337)
strip_active_window_float_ts # (1775008763.0, 1775065348.2999997)


In [ ]:
new_active_window_float_ts = (1776323640.023324, 1776370487.5812337)
new_active_window_float_ts


In [ ]:
region.setRegion(new_active_window_float_ts)


In [ ]:
# timeline.apply_active_window_from_plot_x(*strip_active_window_float_ts, block_signals=False)
# timeline.apply_active_window_from_plot_x(*new_active_window_float_ts, block_signals=True)
timeline.apply_active_window_from_plot_x(*(1775008763.0, 1775065348.2999997), block_signals=True)


In [ ]:
timeline.get_plot_view_range()

In [ ]:
# strip._intervals_item.data
latest_item_ts_float: float = strip._intervals_item.boundingRect().right() ## get right edge timestamp - 1775751404.0
curr_region_width_float: float = np.diff(region.getRegion())[0] # 43214.56095266342
curr_region_width_float

new_window_from_end = [(latest_item_ts_float-curr_region_width_float), latest_item_ts_float]
new_window_from_end

region.setRegion(new_window_from_end)
# strip.set_viewport(new_window_from_end)

In [ ]:
timeline.get_track_names_for_window_sync_group('primary')[0]

a_widget, _, _ = timeline.get_track_tuple(timeline.get_track_names_for_window_sync_group('primary')[0])
a_widget.on_window_changed(start_t=1776323640.023324, end_t=1776370487.5812337)

In [ ]:
motion_widget, motion_track, motion_ds = timeline.get_track_tuple('MOTION_Epoc X Motion')
motion_widget.on_window_changed(start_t=1776323640.023324, end_t=1776370487.5812337)

In [ ]:
region.setAcceptHoverEvents(True)
region.setAcceptTouchEvents(True)

In [ ]:
# region.setRegion()
region.getRegion() # (1776323640.023324, 1776370487.5812337)



In [ ]:
timeline.total_data_end_time

In [ ]:
strip.setAcceptDrops(True)
strip.setInteractive(True)
strip.setMouseTracking(True)
strip.setMouseEnabled(True)

In [ ]:
vb = strip.getViewBox()
vb.setMouseEnabled(False, False)

In [ ]:
a_widget, a_renderer, a_ds = timeline.get_track_tuple('LOG_TextLogger')

In [ ]:
rendered_items = list(a_renderer.detail_graphics.values())[0]

rendered_text_items: list[pg.TextItem] = [v for v in rendered_items if isinstance(v, pg.TextItem)]
rendered_text_items


In [ ]:
for a_text_item in rendered_text_items:
    a_text_item.setAnchor([0.5, 0.5])
    a_text_item.set

In [ ]:
from pypho_timeline.rendering.datasources.specific import EEGTrackDatasource


a_ds: EEGTrackDatasource = timeline.track_datasources['EEG_Epoc X']
a_ds

In [ ]:
a_ds.intervals_df

In [ ]:

timeline = builder.build_from_xdf_file(demo_xdf_path)


In [ ]:
# timeline = main_all_modalities_from_xdf_file_example(demo_xdf_path)

In [ ]:
hasattr(motion_track, "channels_enabled")
motion_track.update_channel_visibility(channel_name='GyroX', is_visible=False)
motion_track.update_channel_visibility(channel_name='GyroY', is_visible=False)
motion_track.update_channel_visibility(channel_name='GyroZ', is_visible=False)

In [ ]:
motion_detail_renderer = motion_track.detail_renderer
motion_detail_renderer.normalize = False

motion_track.datasource.detailed_df
# motion_detail_renderer
# motion_detail_renderer.motion_df


In [ ]:
## INPUTS: timeline
# Get the "EEG Epoc X" track from the timeline
# Fallback to searching tracks by attribute, as there is no `.get_track_by_name` method
# eeg_track = timeline.get_track('EEG_Epoc X')
eeg_widget, eeg_track, eeg_ds = timeline.get_track_tuple('EEG_Epoc X')
detailed_eeg_df: pd.DataFrame = eeg_ds.detailed_df
# print(detailed_eeg_df.columns) # ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4', 't']
detailed_eeg_df




In [ ]:
motion_ds._bad_intervals_unix_df = motion_ds._bad_intervals_unix_df.bad_motion_epochs_df.adding_unix_float_columns(inplace=False)
mask_bad_intervals_df: pd.DataFrame = motion_ds._bad_intervals_unix_df.copy()
mask_bad_intervals_df

In [ ]:
from phopymnehelper.helpers.dataframe_accessor_helpers import MaskedValidDataFrameAccessor

# mask_by_intervals

## find if each detailed_eeg_df['t'] is within the bad interval
# detailed_eeg_df['t']
# Efficiently see if each `detailed_eeg_df['t']` value falls within `mask_bad_intervals_df[['onset', 'onset_end']]` where all columns are datetime64[ns] 

# mask_bad_intervals_df[['onset', 'onset_end']]
# detailed_eeg_df['t'].dtypes # dtype('<M8[ns]')

detailed_eeg_df = detailed_eeg_df.masked_df.masking_by_intervals(mask_bad_intervals_df=mask_bad_intervals_df, time_col_name='t', bool_mask_column_name='is_bad_motion',
                                                            intervals_start_col_name='onset', intervals_end_col_name='onset_end')
detailed_eeg_df.masked_df.add_masking_column(mask_col='is_valid', value_cols=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4'])
detailed_eeg_df


In [ ]:
detailed_eeg_df

In [ ]:
eeg_track.

In [ ]:


masked_detailed_eeg_df: pd.DataFrame = detailed_eeg_df.masked_df.get_masked(copy=True)

# masked_detailed_eeg_df: pd.DataFrame = detailed_eeg_df.masked_df.add_masking_column(mask_col='is_valid', value_cols=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4'],
#                                                      copy=True)
masked_detailed_eeg_df

In [ ]:
masked_detailed_eeg_df.masked_df.get_masked()

In [ ]:
detailed_eeg_df = detailed_eeg_df.drop(columns=['is_bad_motion_right', 'is_bad_motion'], inplace=False)

In [ ]:

# ii = pd.IntervalIndex.from_arrays(
#     mask_bad_intervals_df["onset"],
#     mask_bad_intervals_df["onset_end"],
#     closed="both",  # or "left" / "right" / "neither"
# )
# in_bad = ii.contains(detailed_eeg_df["t"].values)


# import polars as pl

# points = pl.from_pandas(detailed_eeg_df).lazy()  # or scan_csv / DataFrame
# intervals = pl.from_pandas(mask_bad_intervals_df).select("onset", "onset_end").lazy()

# hits = points.join_where(
#     intervals,
#     pl.col("t").is_between(pl.col("onset"), pl.col("onset_end")),
# )

# # timestamps that fall in at least one bad interval
# in_bad = (
#     hits.select("t")
#     .unique()
#     .with_columns(in_bad=pl.lit(True))
# )

# out = (
#     points.join(in_bad, on="t", how="left")
#     .with_columns(pl.col("in_bad").fill_null(False))
# )

# (pl.col("t") >= pl.col("onset")) & (pl.col("t") < pl.col("onset_end"))

import polars as pl

lf = pl.from_pandas(detailed_eeg_df).lazy()
iv = pl.from_pandas(mask_bad_intervals_df).select("onset", "onset_end").lazy()

bad_t = (
    lf.join(iv, how="cross")
    .filter((pl.col("t") >= pl.col("onset")) & (pl.col("t") <= pl.col("onset_end")))
    .select("t")
    .unique()
    .with_columns(pl.lit(True).alias("is_bad_motion"))
)

df = lf.join(bad_t, on="t", how="left").with_columns(
    pl.col("is_bad_motion").fill_null(False)
).collect()

detailed_eeg_df = df.to_pandas()  # if you need pandas again
detailed_eeg_df


In [ ]:
# eeg_track = timeline.get_track('MOTION_Epoc X Motion')

# if hasattr(timeline, "tracks"):
#     for tr in timeline.tracks:
#         # Try 'name' or 'track_name' attribute, fallback to 'datasource.name'
#         track_name = getattr(tr, "name", None) or getattr(tr, "track_name", None)
#         if track_name == "EEG Epoc X":
#             eeg_track = tr
#             break
#         elif hasattr(tr, "datasource") and hasattr(tr.datasource, "name"):
#             if tr.datasource.name == "EEG Epoc X":
#                 eeg_track = tr
#                 break
# eeg_track
eeg_track.channel_visibility

In [ ]:
hasattr(eeg_track, "channels_enabled")
eeg_track.update_channel_visibility(channel_name='GyroX', is_visible=False)


In [ ]:
channels_to_hide = [] # ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
channels_to_hide = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'FC6', 'F4', 'F8', 'AF4'] # , 'O1', 'O2', 'P8', 'T8'
for a_ch in channels_to_hide:
    eeg_track.update_channel_visibility(channel_name=a_ch, is_visible=False)
# eeg_track.update_channel_visibility(channel_name='P8', is_visible=False)


In [ ]:
print(list(eeg_track.channel_visibility.keys()))

In [ ]:
eeg_track.visible_intervals ## this should not be empty


In [ ]:


eeg_track._trigger_visibility_render()

In [ ]:

if eeg_track is not None and hasattr(eeg_track, "channel_visibility"):
    # If there's an attribute listing enabled channels, we'll disable half
    enabled_channels = eeg_track.channel_visibility
    # Otherwise fall back to all channel names if present
    if not enabled_channels and hasattr(eeg_track, "all_channel_names"):
        enabled_channels = eeg_track.all_channel_names
    if enabled_channels:
        # Compute half the channels (disable the second half)
        half = len(enabled_channels) // 2
        # Set only the first half as enabled
        eeg_track.channel_visibility = enabled_channels[:half]
        # If the track supports setting enabled/disabled channels via a method, use it
        if hasattr(eeg_track, 'set_enabled_channels'):
            eeg_track.set_enabled_channels(enabled_channels[:half])
        # Potentially trigger an update/redraw
        if hasattr(eeg_track, 'update_channel_visibility'):
            eeg_track.update_channel_visibility()
else:
    print("Could not find 'EEG Epoc X' track or it doesn't support channel control.")


# Timeline Widget from just Video Track

In [ ]:
enable_video_track: bool = True ## always true for run-video-only mode

In [11]:
from pathlib import Path
from phopylslhelper.file_metadata_caching.manager import BaseFileMetadataManager
from phopylslhelper.file_metadata_caching.video_metadata import VideoMetadataParser
from phopylslhelper.file_metadata_caching.file_metadata import BaseFileMetadataParser

if enable_video_track:
    video_manager: BaseFileMetadataManager = BaseFileMetadataManager(parse_folders=[Path("M:/ScreenRecordings/EyeTrackerVR_Recordings"), Path("M:/ScreenRecordings/REC_continuous_video_recorder")],
                                                                    parsers={'video': VideoMetadataParser},)
    video_manager.metadata_df
    recent_videos = video_manager.get_most_recent_video_paths(max_num_videos=25)

In [12]:
if enable_video_track:
    # Create the VideoTrackDatasource
    video_ds: VideoTrackDatasource = VideoTrackDatasource(video_paths=recent_videos)

    # Choose a name for the new video track
    video_track_name: str = "RecentVideosTrack"

    # Add the new video track to the existing timeline
    # timeline.add_video_track(video_track_name, video_ds)
    video_widget, root_graphics, plot_item, dock = timeline.add_video_track(track_name=video_track_name, video_datasource=video_ds,
                                                                                                                    enable_time_crosshair=True,
                                                                                                                )

# timeline.add_track(video_ds, name=video_track_name)


In [ ]:
from pypho_timeline.timeline_builder import TimelineBuilder
from pypho_timeline.rendering.graphics.track_renderer import TrackRenderer

builder = TimelineBuilder()

# Build a new timeline from just video track (no XDF file needed)
# Option 1: From video folder
# video_timeline = builder.build_from_video(video_folder_path=video_folder)

# Option 2: From list of video paths
video_timeline = builder.build_from_video(video_paths=recent_videos)

# Option 3: From existing VideoTrackDatasource
# video_timeline = builder.build_from_video(video_datasource=video_ds)
video_timeline.show()

In [ ]:
video_timeline.get_all_track_names()

In [ ]:
# video_widget
video_track_name = "VideoTrack"
video_widget, video_track_renderer, video_track_datasource = video_timeline.get_track_tuple(name=video_track_name)
video_widget

In [ ]:
## Add a current datetime red line representing the current datetime (datetime.now())

video_widget.


In [ ]:
video_timeline.show()

In [ ]:
# Create a plot widget for this track
from pypho_timeline.core.synchronized_plot_mode import SynchronizedPlotMode


a_detail_renderer = video_ds.get_detail_renderer()

# The getOptionsPanel() method will be called by the dock when needed
# No need to set optionsPanel attribute if getOptionsPanel() is implemented

## if we provide a valid `optionsPanel: Optional[QWidget]` here on the widget the button will automatically show up on the dock
track_widget, a_root_graphics, a_plot_item, a_dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
    name=video_ds.custom_datasource_name,
    dockSize=(500, 80),
    dockAddLocationOpts=['bottom'],
    sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA
)

assert a_detail_renderer is not None
track_widget.set_track_renderer(a_detail_renderer)
# Explicitly set the attribute (not just rely on getOptionsPanel())
track_widget.optionsPanel = track_widget.getOptionsPanel()

# Try to force dock to update button visibility
a_dock.updateWidgetsHaveOptionsPanel()

a_dock.update()  # May refresh the title bar
# Or if available:
if hasattr(a_dock, 'updateTitleBar') or hasattr(a_dock, 'refresh'):
    a_dock.updateTitleBar()  # or refresh()



# Set the plot to show the full time range
a_plot_item.setXRange(
    timeline.total_data_start_time, 
    timeline.total_data_end_time, 
    padding=0
)
a_plot_item.setYRange(0, 1, padding=0)
a_plot_item.setLabel('bottom', 'Time', units='s')
a_plot_item.setLabel('left', video_ds.custom_datasource_name)
a_plot_item.hideAxis('left')  # Hide Y-axis for cleaner look

# Add the track to the plot
a_track_name: str = video_ds.custom_datasource_name
timeline.add_track(
    video_ds,
    name=a_track_name,
    plot_item=a_plot_item
)

# Messing with stream:

In [ ]:
a_track_name: str = 'MOTION_Epoc X Motion'
# a_track_name: str = 'EEG_Epoc X'
a_renderer = timeline.track_renderers[a_track_name]
a_detail_renderer = a_renderer.detail_renderer # MotionPlotDetailRenderer 
a_ds = timeline.track_datasources[a_track_name]
# interval = a_ds.intervals_df
interval = a_ds.get_overview_intervals()
# interval = None
type(interval)
interval

In [ ]:
a_renderer.visible_intervals
a_renderer.detail_graphics
a_renderer.overview_rects_item



In [ ]:
from pypho_timeline.core.pyqtgraph_time_synchronized_widget import PyqtgraphTimeSynchronizedWidget

dDisplayItem = timeline.ui.dynamic_docked_widget_container.find_display_dock(identifier=a_track_name) # Dock
a_widget: PyqtgraphTimeSynchronizedWidget = timeline.ui.matplotlib_view_widgets[a_track_name] # PyqtgraphTimeSynchronizedWidget 
a_root_graphics_layout_widget = a_widget.getRootGraphicsLayoutWidget()
a_plot_item = a_widget.getRootPlotItem()
a_plot_item


In [ ]:
# Option 2: directly on the PlotItem (delegates to its ViewBox)
(x_min, x_max), (y_min, y_max) = a_plot_item.viewRange()

# The currently visible time window on the track:
t_start, t_end = x_min, x_max

active_viewport_duration_t: float = t_end - t_start
active_viewport_duration_t
(t_start, t_end) # (59086.148423519495, 59102.162193802214)
interval_dict_rep = {'t_start': t_start, 't_duration': active_viewport_duration_t, 't_end': t_end}
interval_dict_rep

In [ ]:
def getOptionsPanel(self):
    """Return the options panel widget for this dock item."""
    from PyQt5.QtWidgets import QWidget, QVBoxLayout, QLabel, QCheckBox, QSpinBox, QGroupBox
    
    options_widget = QWidget()
    main_layout = QVBoxLayout()
    
    # Create a group box for organization
    settings_group = QGroupBox("Settings")
    settings_layout = QVBoxLayout()
    
    # Add your actual settings controls
    self.option_checkbox = QCheckBox("Enable feature")
    settings_layout.addWidget(self.option_checkbox)
    
    self.option_spinbox = QSpinBox()
    self.option_spinbox.setRange(1, 100)
    settings_layout.addWidget(QLabel("Value:"))
    settings_layout.addWidget(self.option_spinbox)
    
    settings_group.setLayout(settings_layout)
    main_layout.addWidget(settings_group)
    main_layout.addStretch()
    
    options_widget.setLayout(main_layout)
    return options_widget

In [ ]:
a_widget._update_plots()

In [ ]:
# a_widget.options_panel
an_options_panel = a_widget.getOptionsPanel()
a_widget.optionsPanel = an_options_panel
a_widget.options_panel.show()




In [ ]:
# def _checkWidgetForOptions(self, widget):
#     """Check if a widget provides an options panel.
    
#     Returns:
#         Optional[QtWidgets.QWidget]: The options panel widget if found, None otherwise.
        
#     A widget can provide options in two ways:
#     1. Implement a method: getOptionsPanel() -> Optional[QWidget]
#     2. Provide an attribute: optionsPanel: Optional[QWidget]
#     """
#     # Check for method
#     if hasattr(widget, 'getOptionsPanel') and callable(getattr(widget, 'getOptionsPanel')):
#         try:
#             options_panel = widget.getOptionsPanel()
#             if options_panel is not None and isinstance(options_panel, QtWidgets.QWidget):
#                 return options_panel
#         except Exception as e:
#             # Silently ignore errors in getOptionsPanel
#             pass
#     # Check for attribute
#     if hasattr(widget, 'optionsPanel'):
#         options_panel = widget.optionsPanel
#         if options_panel is not None and isinstance(options_panel, QtWidgets.QWidget):
#             return options_panel
#     return None



In [ ]:
# Create the options panel
a_widget.optionsPanel = QWidget()
layout = QVBoxLayout()

# Add your options controls here
label = QLabel("Options for this widget")
layout.addWidget(label)

a_widget.optionsPanel.setLayout(layout)


In [ ]:
a_ds.detailed_df
a_ds.enable_downsampling
a_ds.max_points_per_second

a_ds.max_points_per_second


In [ ]:
a_ds.fetch_detailed_data(interval=interval_dict_rep)

In [ ]:
a_widget

In [ ]:
a_widget.active_plot_target


In [ ]:
a_ds.detailed_df

In [ ]:
graphics_objects = a_detail_renderer.render_detail(plot_item=a_plot_item, interval=interval, detail_data=a_ds.detailed_df) # List[PlotDataItem]
graphics_objects

In [ ]:
# clear_detail
a_detail_renderer.clear_detail(plot_item=a_plot_item, graphics_objects=graphics_objects)
a_detail_renderer

In [ ]:
# a_bounds_tuple = a_detail_renderer.get_detail_bounds(interval=a_ds.get_overview_intervals(), detail_data=a_ds.detailed_df) # List[PlotDataItem]
a_bounds_tuple = a_detail_renderer.get_detail_bounds(interval=interval, detail_data=a_ds.detailed_df) # List[PlotDataItem]
print(f'a_bounds_tuple: {a_bounds_tuple}')
x_min, x_max, y_min, y_max = a_bounds_tuple

In [ ]:
timeline.spikes_window.active_time_window
timeline.spikes_window.total_df_start_end_times


In [ ]:
# In your notebook, check:
track_name = 'MOTION_Epoc X Motion'
proxy_key = f'track_viewport_{track_name}'
print(f"Connection exists: {proxy_key in timeline.ui.connections}")

In [ ]:
# Check if sigRangeChanged has connections
viewbox = a_plot_item.getViewBox()
if viewbox is not None:
    print(f"sigRangeChanged connections: {viewbox.sigRangeChanged.receivers}")

In [ ]:
# Get the current viewport and trigger update
viewbox = a_plot_item.getViewBox()
if viewbox is not None:
    x_range, y_range = viewbox.viewRange()
    a_renderer.update_viewport(x_range[0], x_range[1])

## Individual parts of load from xdf file

In [ ]:
xdf_file_path = demo_xdf_path
print("=" * 60)
print(f"pyPhoTimeline - Load all EEG (or LSL) modalities from XDF: {xdf_file_path}")
print("=" * 60)

# Create Qt application
app = pg.mkQApp("pyPhoTimelineXDFExample")

# --- 1. Load the XDF file (using pyxdf) ---
import pyxdf

print(f"Loading XDF file: {xdf_file_path} ...")
streams, file_header = pyxdf.load_xdf(str(xdf_file_path))
print(f"Streams loaded: {[s['info']['name'][0] for s in streams]}")
print(f"File Header: {file_header}")

In [ ]:
all_streams, all_streams_datasources = perform_process_all_streams(streams=streams)

In [ ]:
all_streams_datasources['Epoc X Motion']


In [ ]:
# active_datasources = eeg_datasources
from pypho_timeline import SynchronizedPlotMode


active_datasources_dict = {k:v for k, v in all_streams_datasources.items() if v is not None}
active_datasources = list(active_datasources_dict.values())


# --- 4. Create the timeline widget and add EEG tracks ---
timeline = SimpleTimelineWidget(
    total_start_time=min([ds.total_df_start_end_times[0] for ds in active_datasources]),
    total_end_time=max([ds.total_df_start_end_times[1] for ds in active_datasources]),
    window_duration=10.0,
    window_start_time=min([ds.total_df_start_end_times[0] for ds in active_datasources]),
    add_example_tracks=False  # Don't add example tracks for XDF data
)

# Create plot widgets for each EEG stream and add tracks
for datasource in active_datasources:
    # Create a plot widget for this track
    track_widget, root_graphics, plot_item, dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
        name=datasource.custom_datasource_name,
        dockSize=(500, 80),
        dockAddLocationOpts=['bottom'],
        sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA
    )
    
    # Set the plot to show the full time range
    plot_item.setXRange(
        timeline.total_data_start_time, 
        timeline.total_data_end_time, 
        padding=0
    )
    plot_item.setYRange(0, 1, padding=0)
    plot_item.setLabel('bottom', 'Time', units='s')
    plot_item.setLabel('left', datasource.custom_datasource_name)
    plot_item.hideAxis('left')  # Hide Y-axis for cleaner look
    
    # Add the track to the plot
    timeline.add_track(
        datasource,
        name=datasource.custom_datasource_name,
        plot_item=plot_item
    )

timeline.setWindowTitle(f"pyPhoTimeline - EEG Modalities from XDF: {xdf_file_path.name}")
timeline.resize(1000, 800)
timeline.show()

print("\nTimeline widget created with EEG tracks from XDF:")
for spec_ds in active_datasources:
    print(f"  - {spec_ds.custom_datasource_name}, time: {spec_ds.total_df_start_end_times}")

print("\nScroll on the timeline to see loaded EEG intervals for each stream.")
print("Close the window to exit.\n")


# 2026-03-18 - Smarter cross-stream post-processing

In [ ]:
## cross-stream post-processing
from pypho_timeline.rendering.datasources.specific.eeg import EEGSpectrogramTrackDatasource, EEGTrackDatasource

eeg_ds: EEGTrackDatasource = timeline.track_datasources['EEG_Epoc X']
# eeg_spectogram_ds: EEGSpectrogramTrackDatasource = timeline.track_datasources['EEG_Spectrogram_Epoc X']


In [ ]:
# eeg_spectogram_ds.intervals_df
eeg_spectogram_ds.detailed_df

# 2026-04-15 - Export loaded sessions to EDF for analysis in other apps

In [ ]:
# 2026-04-15 - Export loaded sessions to EDF for analysis in other apps
from pathlib import Path
from typing import List
import mne

## INPUTS: xdf_to_exported_EDF_parent_export_path
assert xdf_to_exported_EDF_parent_export_path is not None
# xdf_to_exported_EDF_parent_export_path = sso.eeg_analyzed_parent_export_path
flat_raws: List[mne.io.Raw] = eeg_ds._flatten_raw_lists_from_dict(eeg_ds.raw_datasets_dict)
edf_export_parent_path: Path = xdf_to_exported_EDF_parent_export_path.resolve() # .joinpath("exported_EDF")
edf_export_parent_path.mkdir(exist_ok=True, parents=True)

exported_edf_paths: List[Path] = []
for raw_idx, a_raw in enumerate(flat_raws):
    source_stem = None
    raw_description = a_raw.info.get("description")
    if raw_description:
        try:
            source_stem = Path(str(raw_description)).stem
        except Exception:
            source_stem = None

    if not source_stem:
        raw_filenames = getattr(a_raw, "filenames", None)
        if raw_filenames and raw_filenames[0]:
            source_stem = Path(str(raw_filenames[0])).stem

    if not source_stem:
        source_stem = f"session_{raw_idx:03d}"

    curr_file_edf_path: Path = edf_export_parent_path.joinpath(f"{source_stem}.edf").resolve()
    exported_edf_paths.append(a_raw.save_to_edf(output_path=curr_file_edf_path))

print(f"Exported {len(exported_edf_paths)} EDF file(s) to: {edf_export_parent_path}")
exported_edf_paths

In [ ]:
from pathlib import Path
from phopymnehelper.xdf_files import XDFDataStreamAccessor, LabRecorderXDF

## INPUTS: loaded session raws in eeg_ds.raw_datasets_dict
post_processed_exports = {}
for a_sess_xdf_filename, eeg_raw_list in eeg_ds.raw_datasets_dict.items():
    session_raws_dict = {DataModalityType.EEG.value: eeg_raw_list}
    session_xdf_path = Path(str(a_sess_xdf_filename))
    eeg_raw, export_filepaths_dict = LabRecorderXDF.save_post_processed_to_fif(raws_dict=session_raws_dict, a_xdf_file=session_xdf_path, labRecorder_PostProcessed_path=sso.eeg_analyzed_parent_export_path.joinpath('LabRecorder_PostProcessed'))
    post_processed_exports[str(session_xdf_path)] = {'eeg_raw': eeg_raw, 'export_filepaths_dict': export_filepaths_dict}

post_processed_exports

In [ ]:
a_raw.save_to_edf(

# 2026-04-24 - Remove black inset border inside dock items

In [ ]:
from pypho_timeline.core.pyqtgraph_time_synchronized_widget import PyqtgraphTimeSynchronizedWidget
from pypho_timeline.widgets.custom_graphics_layout_widget import CustomGraphicsLayoutWidget


a_track_name: str = 'MOTION_Epoc X Motion'
# a_track_name: str = 'EEG_Epoc X'
a_renderer = timeline.track_renderers[a_track_name]
a_detail_renderer = a_renderer.detail_renderer # MotionPlotDetailRenderer 
a_ds = timeline.track_datasources[a_track_name]
# interval = a_ds.intervals_df
interval = a_ds.get_overview_intervals()
# interval = None
type(interval)
interval
a_renderer.visible_intervals
a_renderer.detail_graphics
a_renderer.overview_rects_item


dDisplayItem = timeline.ui.dynamic_docked_widget_container.find_display_dock(identifier=a_track_name) # Dock
a_widget: PyqtgraphTimeSynchronizedWidget = timeline.ui.matplotlib_view_widgets[a_track_name] # PyqtgraphTimeSynchronizedWidget 
a_root_graphics_layout_widget: CustomGraphicsLayoutWidget = a_widget.getRootGraphicsLayoutWidget()
a_plot_item = a_widget.getRootPlotItem()
a_plot_item


In [ ]:
a_plot_item.setAutoFillBackground(True)
a_plot_item.background_color 

In [ ]:
a_root_graphics_layout_widget.

# Capture tracks and config for each track for replication to a different window

In [ ]:
timeline.track_window_sync_groups

In [ ]:
timeline.track_datasources

In [ ]:
from pypho_timeline.widgets.TimelineWindow.MainTimelineWindow import MainTimelineWindow

a_main_window: MainTimelineWindow = timeline.window()
a_main_window.builder


In [ ]:
new_timeline = timeline.create_duplicate_timeline_widget()
secondary_window: MainTimelineWindow = MainTimelineWindow.init_with_timeline(timeline=new_timeline, builder=builder)
secondary_window

In [ ]:


new_timeline = timeline.create_duplicate_timeline_widget()
new_timeline

In [ ]:
new_timeline = timeline.create_child_window()
new_timeline

In [ ]:
from pypho_timeline.widgets.TimelineWindow.MainTimelineWindow import MainTimelineWindow

new_timeline = timeline.create_child_window()
new_timeline
secondary_window: MainTimelineWindow = MainTimelineWindow.init_with_timeline(timeline=new_timeline, builder=builder)
secondary_window


# Computed Dose Curve Tracks from parsing Logs

In [ ]:
from pypho_timeline.rendering.datasources.specific.dose import DoseTrackDatasource

# dose_curve_ds = DoseTrackDatasource.init_from_timeline_text_log_tracks(timeline, source_track_name='LOG_EventBoard')
# dose_curve_ds = DoseTrackDatasource.init_from_timeline_text_log_tracks(timeline)
dose_curve_ds = DoseTrackDatasource.build_dose_curve_track(timeline=timeline, timeline_builder=builder, source_track_name=['LOG_TextLogger', 'LOG_EventBoard'])

Total Number of Quanta 1441
clear_computed_result()
Connected "on_options_changed" signal from options panel to track_renderer.on_options_changed
Connected "on_options_accepted" signal from options panel to track_renderer.on_options_accepted
Connected "on_options_rejected" signal from options panel to track_renderer.on_options_rejected


c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\specific\dose.py:167: UserWarning: Discarding nonzero nanoseconds in conversion.
  sample_time: datetime = dt_val.to_pydatetime() if isinstance(dt_val, pd.Timestamp) else dt_val
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\specific\dose.py:167: UserWarning: Discarding nonzero nanoseconds in conversion.
  sample_time: datetime = dt_val.to_pydatetime() if isinstance(dt_val, pd.Timestamp) else dt_val
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\specific\dose.py:167: UserWarning: Discarding nonzero nanoseconds in conversion.
  sample_time: datetime = dt_val.to_pydatetime() if isinstance(dt_val, pd.Timestamp) else dt_val
C:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\Dose-Analysis-Python\src\dose_analysis_python\Helpers\quantization.py:44: UserWarning: Converting to Period representation will drop timezone informa

LiveWindowEventIntervalMonitoringMixin.on_visible_intervals_changed()
LiveWindowEventIntervalMonitoringMixin.on_visible_event_intervals_added(added_rows: {'DOSE_CURVES_Computed':         t_start  t_duration         t_end
0  1.777510e+09     43200.0  1.777553e+09, 'LOG_TextLogger':         t_start   t_duration         t_end
4  1.777507e+09     0.000000  1.777507e+09
3  1.777509e+09  2591.146026  1.777512e+09
2  1.777525e+09  1952.424541  1.777527e+09
1  1.777546e+09   189.426475  1.777546e+09
0  1.777547e+09   224.930717  1.777547e+09, 'EEG_Spectrogram_Epoc X_Frontal':         t_start   t_duration         t_end
4  1.777507e+09    52.505544  1.777507e+09
3  1.777507e+09  4646.810861  1.777512e+09
2  1.777525e+09  3218.895941  1.777528e+09
1  1.777546e+09   208.510894  1.777546e+09
0  1.777547e+09   367.055280  1.777547e+09, 'EEG_Epoc X':         t_start   t_duration         t_end
4  1.777507e+09    52.505544  1.777507e+09
3  1.777507e+09  4646.810861  1.777512e+09
2  1.777525e+09  3218.8

In [ ]:
# dose_curve_ds.get_detail_renderer()
dose_curve_ds.intervals_df
dose_curve_ds.total_df_start_end_times
dose_curve_ds.recordSeries_df
dose_curve_ds.complete_curve_df


In [ ]:
"""
[2026-04-25 01:23:01] EventBoard: Dose 1.0+ (DOSE_AMPH_30mg)
[2026-04-25 01:44:29] [TRANSCRIBED] Thank  you.
[2026-04-25 01:44:36] [TRANSCRIBED] you.
[2026-04-25 01:45:08] [TRANSCRIBED] Thanks  for  watching!
[2026-04-25 01:45:15] [TRANSCRIBED] Thanks  for  watching!
[2026-04-25 03:52:27] EventBoard: Dose 1.0+ (DOSE_AMPH_30mg)
[2026-04-25 03:52:30] [TRANSCRIBED] 1 .0.
[2026-04-25 03:52:35] [TRANSCRIBED] .0
[2026-04-25 03:52:41] [TRANSCRIBED] Plus.
"""
from dose_analysis_python.FileImportExport.DoseImporter import DoseNoteFragmentParser
from phopymnehelper.MNE_helpers import MNEHelpers
from phopylslhelper.datetime_helpers import to_display_timezone, unix_timestamp_to_datetime, float_to_datetime


all_track_names = timeline.get_all_track_names()
print(f"all_track_names: {all_track_names}")
txt_log_widget, txt_log_renderer, txt_log_ds = timeline.get_track_tuple('LOG_TextLogger')
txt_log_ds

detailed_txt_log_df: pd.DataFrame = deepcopy(txt_log_ds.detailed_df).reset_index(drop=True) #.set_index('t')

if 'dt' not in detailed_txt_log_df.columns:
    detailed_txt_log_df['dt'] = float_to_datetime(detailed_txt_log_df['t'].to_numpy(), reference_datetime=timeline.reference_datetime)

detailed_txt_log_df = detailed_txt_log_df.set_index('dt', drop=False, inplace=False)
## OUTPUTS: detailed_txt_log_df
detailed_txt_log_df

In [ ]:
def parse_message_logs_to_dose_events(detailed_txt_log_df):
    ## Parse message logs for dose-like entries
    ## INPUTS: detailed_txt_log_df
    detailed_txt_log_df['msg']
    detailed_txt_log_df 


    # r'\d+\s*\.?\s*\d*\s*plus'

    # dose_event_pattern = r'\d+\s*\.?\s*\d*\s*plus'
    # matching_rows = detailed_txt_log_df[detailed_txt_log_df['msg'].astype(str).str.contains(dose_event_pattern, regex=True, na=False)]
    # matching_rows

    dose_event_pattern = r'(\d+\s*\.?\s*\d*)\s*plus'

    matched_with_quantity = (
        detailed_txt_log_df.loc[detailed_txt_log_df['msg'].astype(str).str.contains(dose_event_pattern, regex=True, na=False), ['msg']]
        .assign(quantity=lambda df: df['msg'].astype(str).str.extract(dose_event_pattern, expand=False).str.replace(r'\s+', '', regex=True))
    )

    matched_with_quantity

    # matched_with_quantity['dt']

    # a_parsed_dose_record = DoseNoteFragmentParser.parse_numeric_dose([a_parsed_dose])[0]

    parsed_dose_records_dict: Dict[pd.Timestamp, Dict] = dict(zip(matched_with_quantity.index.to_list(), DoseNoteFragmentParser.parse_numeric_dose(matched_with_quantity['quantity'].to_list())))
    parsed_dose_records_dict
    ## OUTPUTS: parsed_dose_records_dict


    ## build records
    default_med_name: str='AMPH'
    recordSeries_df: pd.DataFrame = pd.DataFrame({'recordDoseDate': list(parsed_dose_records_dict.keys()), 
                                                'recordDoseValue': [a_dose_record_dict['value'] for k, a_dose_record_dict in parsed_dose_records_dict.items()],
                                                'modifier': [a_dose_record_dict.get('modifier', '') for k, a_dose_record_dict in parsed_dose_records_dict.items()],
                            })
    recordSeries_df = recordSeries_df.set_index('recordDoseDate', drop=True, inplace=False)
    # recordSeries.index = pd.DatetimeIndex(recordSeries.index).tz_localize("US/Eastern")
    recordSeries_df.index = pd.DatetimeIndex(recordSeries_df.index).tz_convert("US/Eastern")
    recordSeries_df['medication'] = default_med_name
    ## OUTPUTS: recordSeries_df
    return recordSeries_df

## INPUTS: detailed_txt_log_df
recordSeries_df = parse_message_logs_to_dose_events(detailed_txt_log_df)
recordSeries_df

In [ ]:
from datetime import datetime, timedelta
from dose_analysis_python.Helpers.quantization import Quanta, ComputationTimeBlock

## INPUTS: timeline, recordSeries_df
# def build_dose_curve_track(timeline, recordSeries_df):
## compute specific datetime windows by passing the initial/final state vector to the calculation
start_date: datetime = timeline.total_data_start_time # datetime(2026, 4, 13)
# end_date: Optional[datetime] = None
end_date: Optional[datetime] = timeline.total_data_end_time

computation_blocks, complete_curve_df = ComputationTimeBlock.init_from_start_end_date(parsed_record_df=recordSeries_df, start_date=start_date, end_date=end_date)
## OUTPUTS: computation_blocks, complete_curve_df
# print(list(complete_curve_df.columns)) # ['t_h', 'AMPH_gut', 'AMPH_blood', 'AMPH_brain', 'AMPH_ecf', 'DA_str', 'NE_pfc', 'DA_pfc', 'compute_block_idx', 't']
complete_curve_df

In [ ]:
import pandas as pd
from pypho_timeline import SynchronizedPlotMode
from pypho_timeline.rendering.helpers import ChannelNormalizationMode
from pypho_timeline.rendering.datasources.track_datasource import IntervalProvidingTrackDatasource
from pypho_timeline.rendering.detail_renderers.generic_plot_renderer import DataframePlotDetailRenderer


def build_dose_curve_track(timeline, complete_curve_df, track_name = 'DOSE_CURVES_Computed'):
    ## INPUTS: complete_curve_df
    ## time_column_names = ['t_h', 't']
    curve_channel_names = [col for col in ['AMPH_gut', 'AMPH_blood', 'AMPH_brain', 'AMPH_ecf', 'DA_str', 'NE_pfc', 'DA_pfc'] if col in complete_curve_df.columns]
    curve_df = complete_curve_df[['t'] + curve_channel_names].copy()
    interval_start = curve_df['t'].min()
    interval_duration_seconds = (curve_df['t'].max() - interval_start).total_seconds()
    intervals_df = pd.DataFrame({'t_start': [interval_start], 't_duration': [interval_duration_seconds]})
    curve_renderer = DataframePlotDetailRenderer(channel_names=curve_channel_names, normalize=True, 
                                                normalization_mode_dict={
            ('AMPH_gut', 'AMPH_blood', 'AMPH_brain', 'AMPH_ecf'): ChannelNormalizationMode.GROUPMINMAXRANGE,
            ('DA_str', 'DA_pfc'): ChannelNormalizationMode.GROUPMINMAXRANGE,
            ('NE_pfc',): ChannelNormalizationMode.GROUPMINMAXRANGE,
        }, fallback_normalization_mode = ChannelNormalizationMode.INDIVIDUAL,
        pen_width=1.5,
    )

    dose_curve_ds = IntervalProvidingTrackDatasource(intervals_df=intervals_df, detailed_df=curve_df, custom_datasource_name=track_name, detail_renderer=curve_renderer, enable_downsampling=False)

    if track_name in timeline.get_all_track_names():
        print(f"Track '{track_name}' already exists; skipping add.")
    else:
        track_widget, root_graphics, plot_item, dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(name=track_name, dockSize=(500, 120), dockAddLocationOpts=['bottom'], sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA)
        timeline.add_track(dose_curve_ds, name=track_name, plot_item=plot_item)

    
    return dose_curve_ds, curve_renderer



In [ ]:
dose_curve_ds.cl

In [ ]:
from pypho_timeline.rendering.datasources.specific.dose import DoseTrackDatasource


# complete_curve_df, recordSeries_df = DoseTrackDatasource.init_from_timeline_text_log_tracks(timeline=timeline)

dose_curve_ds = DoseTrackDatasource.init_from_timeline_text_log_tracks(timeline=timeline)
dose_curve_ds
# dose_curve_ds, curve_renderer = DoseTrackDatasource.build_dose_curve_track(timeline, recordSeries_df, track_name = 'DOSE_CURVES_Computed')


In [ ]:
from pypho_timeline.rendering.datasources.specific.dose import DoseTrackDatasource

dose_curve_ds = DoseTrackDatasource.build_dose_curve_track(
    timeline=timeline,
    timeline_builder=builder,
    source_track_name='LOG_TextLogger', #'LOG_EventBoard'   # or 'LOG_TextLogger' for other setups
)

In [ ]:
from pypho_timeline.rendering.datasources.specific.dose import DosePlotDetailRenderer

dose_curve_renderer = DosePlotDetailRenderer(text_color='orange', text_size=10, channel_names=modality_channels_dict['LOG'])

datasource = IntervalProvidingTrackDatasource.from_multiple_sources(intervals_dfs=all_intervals_dfs, detailed_dfs=all_detailed_dfs, custom_datasource_name=f"LOG_{stream_name}", detail_renderer=a_detail_renderer, enable_downsampling=False)



In [ ]:
print(list(complete_curve_df.columns)) # ['t_h', 'AMPH_gut', 'AMPH_blood', 'AMPH_brain', 'AMPH_ecf', 'DA_str', 'NE_pfc', 'DA_pfc', 'compute_block_idx', 't']


In [ ]:
# from pypho_timeline.utils.datetime_helpers import float_to_datetime, unix_timestamp_to_datetime, get_reference_datetime_from_xdf_header, to_display_timezone
from phopylslhelper.datetime_helpers import to_display_timezone, unix_timestamp_to_datetime, float_to_datetime

detailed_txt_log_df['t'].to_numpy()

out_dts = unix_timestamp_to_datetime(detailed_txt_log_df['t'].to_numpy())
out_dts
[to_display_timezone(v) for v in out_dts]



In [ ]:


to_display_timezone(np.array(unix_timestamp_to_datetime(detailed_txt_log_df['t'].to_numpy())))

.astimezone(DISPLAY_TIMEZONE)